# Spacecraft domain — high-quality queue generator

Patched version: `v48_quality_diverse_l5`.

This notebook is intentionally queue-based: it first asks WDQS for viable constraint combinations, then fetches full gold lists for accepted candidates, appending records incrementally to JSONL.


In [1]:
# Load common helpers only if this domain notebook is run standalone.
# This avoids `%run ./00_common_helpers.ipynb`, which requires the nbformat package.
from pathlib import Path
if "wd" not in globals() or "rows_from_select" not in globals():
    exec(Path("common_helpers.py").read_text(encoding="utf-8"), globals())


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a id="spacecraft-domain"></a>

## Spacecraft domain

Generates 100–120 complex, diverse Wikidata multi-hop spacecraft questions with stricter deduplication and more diverse L4/L5 patterns.

Design goals:
- no one-criterion L1 questions;
- queue-based generation, no random retry hangs;
- full WDQS gold lists with RU/EN labels;
- clean human-readable constraints with QIDs kept only in metadata/SPARQL;
- rich L4/L5 bridge patterns using seed spacecraft, programs/series, launch vehicles, operators, manufacturers, types, countries, and launch periods.


Patch v47: L1 target is 15, queues are interleaved by template, candidate diversity caps are checked before expensive gold SELECTs, and accepted records store an explicit WDQS count check for gold completeness.

In [2]:

# ============================================================
# Spacecraft v47: high-recall quality queue multi-hop generator
# ============================================================

from __future__ import annotations

import contextlib
import hashlib
import json
import math
import random
import re
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

try:
    import signal
except Exception:  # pragma: no cover
    signal = None

SPACECRAFT_PATCH_VERSION = "v47_recall_multilevel"
SEED = int(globals().get("SEED", 42))

# Base class: Wikidata "spacecraft".
Q_SPACECRAFT = ensure_qid("космический аппарат", fallback_qid="Q40218")
Q_COUNTRY = "Q6256"

# 120 target rows. Final curation can reduce to 100-120 best rows.
SPACECRAFT_TARGET_PER_LEVEL = {
    "L1": 15,
    "L2": 22,
    "L3": 25,
    "L4": 30,
    "L5": 28,
}

# Requested answer count. Harder levels use 3 because real spacecraft data is sparse,
# but the criteria themselves are much more multi-hop.
SPACECRAFT_REQUESTED_COUNT = {
    "L1": 5,
    "L2": 5,
    "L3": 4,
    "L4": 3,
    "L5": 3,
}

# Gold count acceptance: avoid tiny brittle records and very broad noisy records.
SPACECRAFT_ACCEPT_MIN_GOLD = {
    "L1": 7,
    "L2": 6,
    "L3": 5,
    "L4": 4,
    "L5": 3,
}
SPACECRAFT_ACCEPT_MAX_GOLD = {
    "L1": 90,
    "L2": 80,
    "L3": 70,
    "L4": 60,
    "L5": 45,
}

SPACECRAFT_QUERY_LIMIT = 701
SPACECRAFT_CANDIDATE_LIMIT_PER_TEMPLATE = {
    "L1": 300,
    "L2": 380,
    "L3": 420,
    "L4": 520,
    "L5": 650,
}
SPACECRAFT_MAX_CANDIDATES_PER_LEVEL = {
    "L1": 1200,
    "L2": 1800,
    "L3": 2300,
    "L4": 3200,
    "L5": 4200,
}

# Diversity caps. These are deliberately permissive for L4/L5 because the data is sparse.
SPACECRAFT_MAX_SAME_TEMPLATE_PER_LEVEL = {
    "L1": 5,
    "L2": 6,
    "L3": 8,
    "L4": 10,
    "L5": 11,
}
SPACECRAFT_MAX_SAME_PRIMARY_PER_LEVEL = {
    "L1": 5,
    "L2": 6,
    "L3": 7,
    "L4": 8,
    "L5": 9,
}

SPACECRAFT_GOLD_JACCARD_THRESHOLD = 0.90
SPACECRAFT_GOLD_CONTAINMENT_THRESHOLD = 0.96
SPACECRAFT_HARD_QUERY_TIMEOUT_SECONDS = 25
SPACECRAFT_AGG_QUERY_TIMEOUT_SECONDS = 35

# Keep WDQS retries bounded so one bad candidate cannot hang a level for many minutes.
try:
    wd.timeout = min(max(int(getattr(wd, "timeout", 12)), 8), 15)
    wd.max_retries = 1
except Exception:
    pass

# Year buckets used in candidate discovery.
SPACECRAFT_YEAR_RANGES = {
    "broad": [(1957, 1975), (1976, 1995), (1996, 2010), (2011, 2025)],
    "mid": [(1957, 1970), (1971, 1985), (1986, 2000), (2001, 2012), (2013, 2025)],
    "narrow": [(1957, 1965), (1966, 1975), (1976, 1985), (1986, 1995), (1996, 2005), (2006, 2015), (2016, 2025)],
}

OUT_DIR_DOMAIN = Path("out_wikidata_benchmark/domain_outputs")
OUT_DIR_DOMAIN.mkdir(parents=True, exist_ok=True)
SPACECRAFT_OUTPUT_PATH = OUT_DIR_DOMAIN / "spacecraft1.jsonl"
SPACECRAFT_AUDIT_PATH = OUT_DIR_DOMAIN / "spacecraft_generation_audit.json"
SPACECRAFT_CHECKPOINT_PATH = OUT_DIR_DOMAIN / "spacecraft_generation_checkpoint.json"

# Backward-compatible aliases for quick inspection cells.
out_path = SPACECRAFT_OUTPUT_PATH
audit_path = SPACECRAFT_AUDIT_PATH
checkpoint_path = SPACECRAFT_CHECKPOINT_PATH

print("Spacecraft patch:", SPACECRAFT_PATCH_VERSION)
print("output:", SPACECRAFT_OUTPUT_PATH.resolve())


class SpacecraftCandidateTimeout(TimeoutError):
    pass


@contextlib.contextmanager
def _space_time_limit(seconds: int):
    """Hard timeout for a single WDQS operation on Unix/macOS. No-op on unsupported platforms."""
    if signal is None or not hasattr(signal, "SIGALRM"):
        yield
        return

    def handler(signum, frame):  # pragma: no cover
        raise SpacecraftCandidateTimeout(f"WDQS operation exceeded {seconds}s")

    old = signal.signal(signal.SIGALRM, handler)
    signal.alarm(int(seconds))
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def _space_qid(uri_or_qid: str) -> str:
    if not uri_or_qid:
        return ""
    s = str(uri_or_qid)
    if re.fullmatch(r"Q\d+", s):
        return s
    try:
        return uri_to_qid(s)
    except Exception:
        m = re.search(r"Q\d+", s)
        return m.group(0) if m else ""


def _space_int(x: Any) -> Optional[int]:
    if x is None:
        return None
    try:
        return int(float(str(x)))
    except Exception:
        return None


def _space_lbl(row: Dict[str, Any], base: str, lang: str = "en") -> str:
    key = f"{base}Label{lang.upper()[0] + lang[1:]}"
    if key in row and row.get(key):
        return str(row[key])
    # Common explicit key variants used below.
    if lang == "en":
        for k in (f"{base}LabelEn", f"{base}Label", base):
            if row.get(k):
                return str(row[k])
    if lang == "ru":
        for k in (f"{base}LabelRu", f"{base}LabelEn", f"{base}Label", base):
            if row.get(k):
                return str(row[k])
    return ""


def _space_year_values(kind: str = "broad") -> str:
    return "\n".join(f"        ({y1} {y2})" for y1, y2 in SPACECRAFT_YEAR_RANGES[kind])


def _space_year_filter(y1_var: str = "?y1", y2_var: str = "?y2") -> List[str]:
    return [
        "?item wdt:P619 ?launch_date .",
        "BIND(YEAR(?launch_date) AS ?launch_year) .",
        f"FILTER(?launch_year >= {y1_var} && ?launch_year <= {y2_var}) .",
    ]


def _space_fixed_year_filter(y1: int, y2: int) -> List[str]:
    return [
        "?item wdt:P619 ?launch_date .",
        "BIND(YEAR(?launch_date) AS ?launch_year) .",
        f"FILTER(?launch_year >= {int(y1)} && ?launch_year <= {int(y2)}) .",
    ]


def _space_label_lines(var: str, name: str) -> str:
    return (
        f"{var} rdfs:label ?{name}LabelEn FILTER(LANG(?{name}LabelEn) = \"en\") .\n"
        f"      OPTIONAL {{ {var} rdfs:label ?{name}LabelRu FILTER(LANG(?{name}LabelRu) = \"ru\") . }}"
    )


def _space_rows_from_sparql(sparql: str, timeout_seconds: int) -> List[Dict[str, Any]]:
    with _space_time_limit(timeout_seconds):
        return rows_from_select(wd.sparql_select(sparql))


def _space_group_having(complexity: str) -> str:
    mn = SPACECRAFT_ACCEPT_MIN_GOLD[complexity]
    mx = SPACECRAFT_ACCEPT_MAX_GOLD[complexity]
    return f"HAVING(COUNT(DISTINCT ?item) >= {mn} && COUNT(DISTINCT ?item) <= {mx})"


def _space_count_from_row(row: Dict[str, Any]) -> int:
    return _space_int(row.get("cnt")) or 0


def _space_candidate_sort_key(c: Dict[str, Any]) -> Tuple[int, int, str]:
    # Prefer medium gold counts and diverse templates. Extremely broad candidates are less useful.
    cnt = int(c.get("expected_count") or 0)
    k = SPACECRAFT_REQUESTED_COUNT[c["complexity"]]
    target = max(k + 4, 10)
    return (abs(cnt - target), cnt, c.get("template_id", ""))


def _space_build_ask(where_lines: Sequence[str]) -> str:
    body = "\n      ".join(where_lines)
    return f"""# WDQS-only validator. Replace {{ITEM}} with a candidate spacecraft QID.
ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?item)
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} .
      {body}
}}"""


def _space_fetch_golds(where_lines: Sequence[str], limit: int = SPACECRAFT_QUERY_LIMIT) -> Tuple[str, List[Dict[str, str]]]:
    where = "\n      ".join(where_lines)
    sparql = f"""
    SELECT DISTINCT ?item ?itemLabelEn ?itemLabelRu WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} .
      {where}
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
    }}
    LIMIT {int(limit)}
    """.strip()
    rows = _space_rows_from_sparql(sparql, SPACECRAFT_HARD_QUERY_TIMEOUT_SECONDS)
    seen = set()
    out: List[Dict[str, str]] = []
    for r in rows:
        qid = _space_qid(r.get("item", ""))
        en = str(r.get("itemLabelEn") or "").strip()
        ru = str(r.get("itemLabelRu") or en).strip()
        if qid and en and qid not in seen:
            seen.add(qid)
            out.append({"qid": qid, "label_en": en, "label_ru": ru})
    return sparql, out



def _space_count_matching_items(where_lines: Sequence[str], require_en_label: bool = True) -> int:
    """Count all items matching the exact validator constraints.

    By default this uses the same English-label requirement as the gold SELECT, so the
    stored gold list can honestly be interpreted as complete for the benchmark's
    answer universe (items with an English label). A secondary no-label count is stored
    in metadata only as a diagnostic signal.
    """
    where = "\n      ".join(where_lines)
    label_clause = '?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .' if require_en_label else ''
    sparql = f"""
    SELECT (COUNT(DISTINCT ?item) AS ?cnt) WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} .
      {where}
      {label_clause}
    }}
    """.strip()
    rows = _space_rows_from_sparql(sparql, SPACECRAFT_HARD_QUERY_TIMEOUT_SECONDS)
    return _space_int((rows[0] if rows else {}).get("cnt")) or 0


def _space_interleave_candidates(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Interleave candidates by template_id so early accepted rows are diverse."""
    buckets: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for r in sorted(records, key=_space_candidate_sort_key):
        buckets[str(r.get("template_id", ""))].append(r)
    out: List[Dict[str, Any]] = []
    keys = sorted(buckets)
    while keys:
        next_keys = []
        for k in keys:
            if buckets[k]:
                out.append(buckets[k].pop(0))
            if buckets[k]:
                next_keys.append(k)
        keys = next_keys
    return out


def _space_should_try_candidate(cand: Dict[str, Any], level: str, existing_constraint_sigs: set, counters: Dict[str, Counter]) -> Tuple[bool, str]:
    """Cheap pre-filter before running the expensive per-candidate gold SELECT."""
    constraints = _space_clean_constraints(cand.get("constraints", {}))
    sig = _space_sig_constraints(constraints)
    if sig in existing_constraint_sigs:
        return False, "duplicate_constraints_pre"
    template_id = cand.get("template_id", "")
    if counters["template_level"][(level, template_id)] >= SPACECRAFT_MAX_SAME_TEMPLATE_PER_LEVEL[level]:
        return False, "template_cap_pre"
    primary = _space_primary_key({"constraints": constraints})
    if counters["primary_level"][(level, primary)] >= SPACECRAFT_MAX_SAME_PRIMARY_PER_LEVEL[level]:
        return False, "primary_constraint_cap_pre"
    return True, "ok"

def _space_clean_constraints(d: Dict[str, Any]) -> Dict[str, Any]:
    # Constraints must be clean and human-readable: no QIDs, no property names, no wd: terms.
    clean = {"kind": "spacecraft"}
    for k, v in d.items():
        if v is None or v == "":
            continue
        if k.endswith("_qid") or k.endswith("_ru") or k.endswith("_en"):
            continue
        if isinstance(v, str):
            if re.fullmatch(r"Q\d+", v) or v.startswith("wd:") or re.fullmatch(r"P\d+", v):
                continue
        clean[k] = v
    return clean


def _space_sig_constraints(c: Dict[str, Any]) -> str:
    return json.dumps(c, ensure_ascii=False, sort_keys=True)


def _space_gold_set(r: Dict[str, Any]) -> set:
    return set(r.get("gold_answer_qids") or [])


def _space_jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def _space_containment(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / min(len(a), len(b))


def _space_primary_key(r_or_c: Dict[str, Any]) -> str:
    c = r_or_c.get("constraints", r_or_c)
    for key in [
        "operator", "manufacturer", "country_of_origin", "type", "launch_vehicle", "program_or_series",
        "same_program_as", "same_operator_as", "same_manufacturer_as", "same_launch_vehicle_as", "same_type_as", "same_country_of_origin_as",
    ]:
        if c.get(key):
            return f"{key}:{c[key]}"
    return "misc"


def _space_record_key_order(r: Dict[str, Any]) -> Dict[str, Any]:
    keys = [
        "id", "domain", "complexity", "query_text_ru", "constraints", "requested_count",
        "gold_answer_qids", "gold_answer_labels_ru", "sparql_query", "created_at",
        "query_text_en", "gold_answer_labels_en", "is_advanced", "template_id", "template_family",
        "gold_truncated", "ask_validator_sparql", "local_validator", "gold_collection_meta",
        "gold_answer_imdb_ids", "gold_answer_imdb_titles",
    ]
    return {k: r.get(k) for k in keys}


def _space_make_record(cand: Dict[str, Any], idx: int) -> Optional[Dict[str, Any]]:
    complexity = cand["complexity"]
    k = SPACECRAFT_REQUESTED_COUNT[complexity]
    try:
        sparql, golds = _space_fetch_golds(cand["where_lines"], limit=SPACECRAFT_QUERY_LIMIT)
    except SpacecraftCandidateTimeout as e:
        cand["skip_reason"] = f"timeout: {e}"
        return None
    except Exception as e:
        cand["skip_reason"] = f"sparql_error: {type(e).__name__}: {e}"
        return None

    n = len(golds)
    try:
        complete_count_with_en_label = _space_count_matching_items(cand["where_lines"], require_en_label=True)
    except Exception:
        complete_count_with_en_label = n
    try:
        diagnostic_count_without_label_filter = _space_count_matching_items(cand["where_lines"], require_en_label=False)
    except Exception:
        diagnostic_count_without_label_filter = None
    if complete_count_with_en_label > n and n < SPACECRAFT_QUERY_LIMIT:
        # This should be rare; keep the record conservative and mark it as incomplete instead of accepting it silently.
        cand["skip_reason"] = f"gold_count_mismatch:{n}_vs_{complete_count_with_en_label}"
        return None
    if n < max(k, SPACECRAFT_ACCEPT_MIN_GOLD[complexity]):
        cand["skip_reason"] = f"not_enough_gold:{n}"
        return None
    if n > SPACECRAFT_ACCEPT_MAX_GOLD[complexity]:
        cand["skip_reason"] = f"too_many_gold:{n}"
        return None
    if n >= SPACECRAFT_QUERY_LIMIT:
        cand["skip_reason"] = "query_limit_truncated"
        return None

    constraints = _space_clean_constraints(cand["constraints"])
    local_validator = {
        "type": "none_wdqs_only",
        "source": "Wikidata Query Service",
        "applies_after": "ask_validator_sparql",
        "filters": constraints,
        "label_matching_used": False,
        "note": "All constraints for this spacecraft task are represented directly in the WDQS ASK validator; no external local validator is required.",
    }
    meta = {
        "source": "wikidata_sparql",
        "wdqs_candidate_limit": SPACECRAFT_QUERY_LIMIT,
        "rows_returned_by_wdqs": n,
        "gold_returned_before_limits": n,
        "dropped_no_qid_count": 0,
        "dropped_no_en_label_count": 0,
        "label_sources": {
            "ru_label": sum(1 for g in golds if g["label_ru"] != g["label_en"]),
            "en_fallback_for_ru": sum(1 for g in golds if g["label_ru"] == g["label_en"]),
        },
        "gold_may_be_incomplete_due_to_wdqs_limit": False,
        "quality_filter_applied": True,
        "constraints_are_wdqs_only": True,
        "constraints": constraints,
        "gold_limit": 300,
        "gold_returned": n,
        "gold_total_before_limit": complete_count_with_en_label,
        "complete_count_with_en_label": complete_count_with_en_label,
        "diagnostic_count_without_label_filter": diagnostic_count_without_label_filter,
        "gold_truncated_by_local_limit": False,
        "template_id": cand["template_id"],
        "template_family": cand["template_family"],
        "patch_version": SPACECRAFT_PATCH_VERSION,
        "expected_count_from_candidate_query": cand.get("expected_count"),
        "bridge_meta": cand.get("bridge_meta", {}),
        "candidate_constraints_signature": _space_sig_constraints(constraints),
    }
    rec = {
        "id": f"spacecraft_{complexity.lower()}_{idx:04d}",
        "domain": "spacecraft",
        "complexity": complexity,
        "query_text_ru": cand["query_text_ru"],
        "constraints": constraints,
        "requested_count": k,
        "gold_answer_qids": [g["qid"] for g in golds],
        "gold_answer_labels_ru": [g["label_ru"] for g in golds],
        "sparql_query": sparql,
        "created_at": utc_now_z(),
        "query_text_en": cand["query_text_en"],
        "gold_answer_labels_en": [g["label_en"] for g in golds],
        "is_advanced": complexity in {"L3", "L4", "L5"},
        "template_id": cand["template_id"],
        "template_family": cand["template_family"],
        "gold_truncated": False,
        "ask_validator_sparql": _space_build_ask(cand["where_lines"]),
        "local_validator": local_validator,
        "gold_collection_meta": meta,
        "gold_answer_imdb_ids": [],
        "gold_answer_imdb_titles": [],
    }
    return _space_record_key_order(rec)


def _space_add_template_rows(
    out: List[Dict[str, Any]],
    complexity: str,
    template_id: str,
    template_family: str,
    sparql: str,
    row_to_candidate: Callable[[Dict[str, Any]], Optional[Dict[str, Any]]],
) -> None:
    try:
        rows = _space_rows_from_sparql(sparql, SPACECRAFT_AGG_QUERY_TIMEOUT_SECONDS)
    except Exception as e:
        out.append({
            "complexity": complexity,
            "template_id": template_id,
            "template_family": template_family,
            "candidate_error": f"{type(e).__name__}: {e}",
        })
        return
    for row in rows:
        try:
            cand = row_to_candidate(row)
            if not cand:
                continue
            cand.setdefault("complexity", complexity)
            cand.setdefault("template_id", template_id)
            cand.setdefault("template_family", template_family)
            cand.setdefault("expected_count", _space_count_from_row(row))
            out.append(cand)
        except Exception:
            continue


def _space_group_query(
    complexity: str,
    select_vars: str,
    where: str,
    group_vars: str,
    limit: Optional[int] = None,
) -> str:
    if limit is None:
        limit = SPACECRAFT_CANDIDATE_LIMIT_PER_TEMPLATE[complexity]
    having = _space_group_having(complexity)
    return f"""
    SELECT {select_vars} (COUNT(DISTINCT ?item) AS ?cnt)
    WHERE {{
      ?item wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} .
      ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .
      {where}
    }}
    GROUP BY {group_vars}
    {having}
    LIMIT {int(limit)}
    """.strip()


# ============================================================
# Candidate queues by level
# ============================================================

def build_spacecraft_candidates_l1() -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    years = _space_year_values("broad")
    complexity = "L1"

    # type + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P31/wdt:P279* ?type ; wdt:P619 ?launch_date .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?type', 'type')}
    """
    sparql = _space_group_query(complexity, "?type ?typeLabelEn ?typeLabelRu ?y1 ?y2", where, "?type ?typeLabelEn ?typeLabelRu ?y1 ?y2")
    def make_type_period(r):
        type_qid, type_en, type_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([type_qid, type_en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P31/wdt:P279* wd:{type_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 5 космических аппаратов типа «{type_ru}», запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 5 spacecraft of type {type_en} launched between {y1} and {y2}.",
            "constraints": {"type": type_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"type_qid": type_qid, "type_label_en": type_en, "type_label_ru": type_ru},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l1_type_launch_period", "type_period", sparql, make_type_period)

    # operator + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P137 ?operator ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?operator', 'operator')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?y1 ?y2", where, "?operator ?operatorLabelEn ?operatorLabelRu ?y1 ?y2")
    def make_operator_period(r):
        qid, en, ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([qid, en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 5 космических аппаратов, оператором которых является «{ru}» и которые были запущены в период {y1}–{y2} годов.",
            "query_text_en": f"Name 5 spacecraft operated by {en} and launched between {y1} and {y2}.",
            "constraints": {"operator": en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"operator_qid": qid, "operator_label_en": en, "operator_label_ru": ru},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l1_operator_launch_period", "operator_period", sparql, make_operator_period)

    # manufacturer + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P176 ?manufacturer ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?manufacturer', 'manufacturer')}
    """
    sparql = _space_group_query(complexity, "?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2", where, "?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2")
    def make_manufacturer_period(r):
        qid, en, ru = _space_qid(r.get("manufacturer")), _space_lbl(r, "manufacturer", "en"), _space_lbl(r, "manufacturer", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([qid, en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P176 wd:{qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 5 космических аппаратов, произведённых организацией «{ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 5 spacecraft manufactured by {en} and launched between {y1} and {y2}.",
            "constraints": {"manufacturer": en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"manufacturer_qid": qid, "manufacturer_label_en": en, "manufacturer_label_ru": ru},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l1_manufacturer_launch_period", "manufacturer_period", sparql, make_manufacturer_period)

    # launch vehicle + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P375 ?launch_vehicle ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2", where, "?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2")
    def make_lv_period(r):
        qid, en, ru = _space_qid(r.get("launch_vehicle")), _space_lbl(r, "launch_vehicle", "en"), _space_lbl(r, "launch_vehicle", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([qid, en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P375 wd:{qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 5 космических аппаратов, запущенных ракетой-носителем «{ru}» в период {y1}–{y2} годов.",
            "query_text_en": f"Name 5 spacecraft launched by {en} between {y1} and {y2}.",
            "constraints": {"launch_vehicle": en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"launch_vehicle_qid": qid, "launch_vehicle_label_en": en, "launch_vehicle_label_ru": ru},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l1_launch_vehicle_launch_period", "launch_vehicle_period", sparql, make_lv_period)

    # country of origin + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P495 ?country ; wdt:P619 ?launch_date .
      ?country wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?country', 'country')}
    """
    sparql = _space_group_query(complexity, "?country ?countryLabelEn ?countryLabelRu ?y1 ?y2", where, "?country ?countryLabelEn ?countryLabelRu ?y1 ?y2")
    def make_country_period(r):
        qid, en, ru = _space_qid(r.get("country")), _space_lbl(r, "country", "en"), _space_lbl(r, "country", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([qid, en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P495 wd:{qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 5 космических аппаратов страны происхождения «{ru}», запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 5 spacecraft whose country of origin is {en} and that were launched between {y1} and {y2}.",
            "constraints": {"country_of_origin": en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"country_qid": qid, "country_label_en": en, "country_label_ru": ru},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l1_country_launch_period", "country_period", sparql, make_country_period)

    records = [r for r in records if "candidate_error" not in r]
    records.sort(key=_space_candidate_sort_key)
    return records[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL[complexity]]


def build_spacecraft_candidates_l2() -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    complexity = "L2"

    # operator + manufacturer
    where = f"""
      ?item wdt:P137 ?operator ; wdt:P176 ?manufacturer .
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu", where, "?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu")
    def make_op_manu(r):
        op_qid, op_en, op_ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        manu_qid, manu_en, manu_ru = _space_qid(r.get("manufacturer")), _space_lbl(r, "manufacturer", "en"), _space_lbl(r, "manufacturer", "ru")
        if not all([op_qid, op_en, manu_qid, manu_en]): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P176 wd:{manu_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов, оператором которых является «{op_ru}» и которые произведены организацией «{manu_ru}».",
            "query_text_en": f"Name 5 spacecraft operated by {op_en} and manufactured by {manu_en}.",
            "constraints": {"operator": op_en, "manufacturer": manu_en},
            "bridge_meta": {"operator_qid": op_qid, "manufacturer_qid": manu_qid, "operator_label_en": op_en, "manufacturer_label_en": manu_en},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_operator_manufacturer", "operator_manufacturer", sparql, make_op_manu)

    # operator + type
    where = f"""
      ?item wdt:P137 ?operator ; wdt:P31/wdt:P279* ?type .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?type', 'type')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?type ?typeLabelEn ?typeLabelRu", where, "?operator ?operatorLabelEn ?operatorLabelRu ?type ?typeLabelEn ?typeLabelRu")
    def make_op_type(r):
        op_qid, op_en, op_ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        type_qid, type_en, type_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        if not all([op_qid, op_en, type_qid, type_en]): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P31/wdt:P279* wd:{type_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов типа «{type_ru}», оператором которых является «{op_ru}».",
            "query_text_en": f"Name 5 spacecraft of type {type_en} operated by {op_en}.",
            "constraints": {"operator": op_en, "type": type_en},
            "bridge_meta": {"operator_qid": op_qid, "type_qid": type_qid, "operator_label_en": op_en, "type_label_en": type_en},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_operator_type", "operator_type", sparql, make_op_type)

    # manufacturer + type
    where = f"""
      ?item wdt:P176 ?manufacturer ; wdt:P31/wdt:P279* ?type .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?type', 'type')}
    """
    sparql = _space_group_query(complexity, "?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?type ?typeLabelEn ?typeLabelRu", where, "?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?type ?typeLabelEn ?typeLabelRu")
    def make_manu_type(r):
        manu_qid, manu_en, manu_ru = _space_qid(r.get("manufacturer")), _space_lbl(r, "manufacturer", "en"), _space_lbl(r, "manufacturer", "ru")
        type_qid, type_en, type_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        if not all([manu_qid, manu_en, type_qid, type_en]): return None
        return {
            "where_lines": [f"?item wdt:P176 wd:{manu_qid} .", f"?item wdt:P31/wdt:P279* wd:{type_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов типа «{type_ru}», произведённых организацией «{manu_ru}».",
            "query_text_en": f"Name 5 spacecraft of type {type_en} manufactured by {manu_en}.",
            "constraints": {"manufacturer": manu_en, "type": type_en},
            "bridge_meta": {"manufacturer_qid": manu_qid, "type_qid": type_qid, "manufacturer_label_en": manu_en, "type_label_en": type_en},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_manufacturer_type", "manufacturer_type", sparql, make_manu_type)

    # launch vehicle + type
    where = f"""
      ?item wdt:P375 ?launch_vehicle ; wdt:P31/wdt:P279* ?type .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
      {_space_label_lines('?type', 'type')}
    """
    sparql = _space_group_query(complexity, "?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?type ?typeLabelEn ?typeLabelRu", where, "?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?type ?typeLabelEn ?typeLabelRu")
    def make_lv_type(r):
        lv_qid, lv_en, lv_ru = _space_qid(r.get("launch_vehicle")), _space_lbl(r, "launch_vehicle", "en"), _space_lbl(r, "launch_vehicle", "ru")
        type_qid, type_en, type_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        if not all([lv_qid, lv_en, type_qid, type_en]): return None
        return {
            "where_lines": [f"?item wdt:P375 wd:{lv_qid} .", f"?item wdt:P31/wdt:P279* wd:{type_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов типа «{type_ru}», запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 5 spacecraft of type {type_en} launched by {lv_en}.",
            "constraints": {"launch_vehicle": lv_en, "type": type_en},
            "bridge_meta": {"launch_vehicle_qid": lv_qid, "type_qid": type_qid, "launch_vehicle_label_en": lv_en, "type_label_en": type_en},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_launch_vehicle_type", "launch_vehicle_type", sparql, make_lv_type)

    # country + type + launch period (3 criteria, still L2)
    years = _space_year_values("mid")
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P495 ?country ; wdt:P31/wdt:P279* ?type ; wdt:P619 ?launch_date .
      ?country wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?country', 'country')}
      {_space_label_lines('?type', 'type')}
    """
    sparql = _space_group_query(complexity, "?country ?countryLabelEn ?countryLabelRu ?type ?typeLabelEn ?typeLabelRu ?y1 ?y2", where, "?country ?countryLabelEn ?countryLabelRu ?type ?typeLabelEn ?typeLabelRu ?y1 ?y2")
    def make_country_type_period(r):
        c_qid, c_en, c_ru = _space_qid(r.get("country")), _space_lbl(r, "country", "en"), _space_lbl(r, "country", "ru")
        t_qid, t_en, t_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([c_qid, c_en, t_qid, t_en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P495 wd:{c_qid} .", f"?item wdt:P31/wdt:P279* wd:{t_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 5 космических аппаратов типа «{t_ru}» страны происхождения «{c_ru}», запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 5 spacecraft of type {t_en} whose country of origin is {c_en} and that were launched between {y1} and {y2}.",
            "constraints": {"country_of_origin": c_en, "type": t_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"country_qid": c_qid, "type_qid": t_qid, "country_label_en": c_en, "type_label_en": t_en},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_country_type_launch_period", "country_type_period", sparql, make_country_type_period)

    records = [r for r in records if "candidate_error" not in r]
    records.sort(key=_space_candidate_sort_key)
    return records[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL[complexity]]


def build_spacecraft_candidates_l3() -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    complexity = "L3"
    years = _space_year_values("mid")

    # operator + manufacturer + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2", where, "?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2")
    def make_op_manu_period(r):
        op_qid, op_en, op_ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        manu_qid, manu_en, manu_ru = _space_qid(r.get("manufacturer")), _space_lbl(r, "manufacturer", "en"), _space_lbl(r, "manufacturer", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([op_qid, op_en, manu_qid, manu_en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P176 wd:{manu_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 4 космических аппарата, оператором которых является «{op_ru}», произведённых организацией «{manu_ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 4 spacecraft operated by {op_en}, manufactured by {manu_en}, and launched between {y1} and {y2}.",
            "constraints": {"operator": op_en, "manufacturer": manu_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"operator_qid": op_qid, "manufacturer_qid": manu_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l3_operator_manufacturer_launch_period", "operator_manufacturer_period", sparql, make_op_manu_period)

    # operator + type + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P137 ?operator ; wdt:P31/wdt:P279* ?type ; wdt:P619 ?launch_date .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?type', 'type')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?type ?typeLabelEn ?typeLabelRu ?y1 ?y2", where, "?operator ?operatorLabelEn ?operatorLabelRu ?type ?typeLabelEn ?typeLabelRu ?y1 ?y2")
    def make_op_type_period(r):
        op_qid, op_en, op_ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        t_qid, t_en, t_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([op_qid, op_en, t_qid, t_en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P31/wdt:P279* wd:{t_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 4 космических аппарата типа «{t_ru}», оператором которых является «{op_ru}» и которые были запущены в период {y1}–{y2} годов.",
            "query_text_en": f"Name 4 spacecraft of type {t_en}, operated by {op_en}, and launched between {y1} and {y2}.",
            "constraints": {"operator": op_en, "type": t_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"operator_qid": op_qid, "type_qid": t_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l3_operator_type_launch_period", "operator_type_period", sparql, make_op_type_period)

    # launch vehicle + operator + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P375 ?launch_vehicle ; wdt:P137 ?operator ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
      {_space_label_lines('?operator', 'operator')}
    """
    sparql = _space_group_query(complexity, "?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?y1 ?y2", where, "?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?y1 ?y2")
    def make_lv_op_period(r):
        lv_qid, lv_en, lv_ru = _space_qid(r.get("launch_vehicle")), _space_lbl(r, "launch_vehicle", "en"), _space_lbl(r, "launch_vehicle", "ru")
        op_qid, op_en, op_ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([lv_qid, lv_en, op_qid, op_en, y1, y2]): return None
        return {
            "where_lines": [f"?item wdt:P375 wd:{lv_qid} .", f"?item wdt:P137 wd:{op_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 4 космических аппарата, запущенных ракетой-носителем «{lv_ru}», оператором которых является «{op_ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 4 spacecraft launched by {lv_en}, operated by {op_en}, and launched between {y1} and {y2}.",
            "constraints": {"launch_vehicle": lv_en, "operator": op_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"launch_vehicle_qid": lv_qid, "operator_qid": op_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l3_launch_vehicle_operator_launch_period", "launch_vehicle_operator_period", sparql, make_lv_op_period)

    # program/series + launch vehicle
    where = f"""
      ?item wdt:P361 ?program ; wdt:P375 ?launch_vehicle .
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?program ?programLabelEn ?programLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?program ?programLabelEn ?programLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_program_lv(r):
        p_qid, p_en, p_ru = _space_qid(r.get("program")), _space_lbl(r, "program", "en"), _space_lbl(r, "program", "ru")
        lv_qid, lv_en, lv_ru = _space_qid(r.get("launch_vehicle")), _space_lbl(r, "launch_vehicle", "en"), _space_lbl(r, "launch_vehicle", "ru")
        if not all([p_qid, p_en, lv_qid, lv_en]): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P375 wd:{lv_qid} ."],
            "query_text_ru": f"Назови 4 космических аппарата из программы/серии «{p_ru}», запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 4 spacecraft that are part of the {p_en} program/series and were launched by {lv_en}.",
            "constraints": {"program_or_series": p_en, "launch_vehicle": lv_en},
            "bridge_meta": {"program_qid": p_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l3_program_launch_vehicle", "program_launch_vehicle", sparql, make_program_lv)

    records = [r for r in records if "candidate_error" not in r]
    records.sort(key=_space_candidate_sort_key)
    return records[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL[complexity]]


def build_spacecraft_candidates_l4() -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    complexity = "L4"
    years = _space_year_values("narrow")

    # Same program as seed + launch period.
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P361 ?program .
      ?item wdt:P361 ?program ; wdt:P619 ?launch_date .
      FILTER(?item != ?seed)
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?program', 'program')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?program ?programLabelEn ?programLabelRu ?y1 ?y2", where, "?seed ?seedLabelEn ?seedLabelRu ?program ?programLabelEn ?programLabelRu ?y1 ?y2")
    def make_same_program_period(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get("seed")), _space_lbl(r, "seed", "en"), _space_lbl(r, "seed", "ru")
        p_qid, p_en, p_ru = _space_qid(r.get("program")), _space_lbl(r, "program", "en"), _space_lbl(r, "program", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([seed_qid, seed_en, p_qid, p_en, y1, y2]): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P361 ?program .", "?item wdt:P361 ?program .", "FILTER(?item != ?seed) .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата из той же программы/серии, что и «{seed_ru}», запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft from the same program/series as {seed_en} that were launched between {y1} and {y2}.",
            "constraints": {"same_program_as": seed_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"seed_qid": seed_qid, "seed_label_en": seed_en, "program_qid": p_qid, "program_label_en": p_en},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_same_program_as_seed_launch_period", "same_program_seed_period", sparql, make_same_program_period)

    # Same operator as seed + manufacturer + launch period.
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P137 ?operator .
      ?item wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P619 ?launch_date .
      FILTER(?item != ?seed)
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2", where, "?seed ?seedLabelEn ?seedLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2")
    def make_same_op_manu_period(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get("seed")), _space_lbl(r, "seed", "en"), _space_lbl(r, "seed", "ru")
        op_qid, op_en = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en")
        manu_qid, manu_en, manu_ru = _space_qid(r.get("manufacturer")), _space_lbl(r, "manufacturer", "en"), _space_lbl(r, "manufacturer", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([seed_qid, seed_en, op_qid, op_en, manu_qid, manu_en, y1, y2]): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P137 ?operator .", "?item wdt:P137 ?operator .", f"?item wdt:P176 wd:{manu_qid} .", "FILTER(?item != ?seed) .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата с тем же оператором, что и «{seed_ru}», произведённых организацией «{manu_ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft with the same operator as {seed_en}, manufactured by {manu_en}, and launched between {y1} and {y2}.",
            "constraints": {"same_operator_as": seed_en, "manufacturer": manu_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"seed_qid": seed_qid, "operator_qid": op_qid, "operator_label_en": op_en, "manufacturer_qid": manu_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_same_operator_as_seed_manufacturer_launch_period", "same_operator_seed_manufacturer_period", sparql, make_same_op_manu_period)

    # Same launch vehicle as seed + operator + launch period.
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P375 ?launch_vehicle .
      ?item wdt:P375 ?launch_vehicle ; wdt:P137 ?operator ; wdt:P619 ?launch_date .
      FILTER(?item != ?seed)
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
      {_space_label_lines('?operator', 'operator')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?y1 ?y2", where, "?seed ?seedLabelEn ?seedLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?y1 ?y2")
    def make_same_lv_op_period(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get("seed")), _space_lbl(r, "seed", "en"), _space_lbl(r, "seed", "ru")
        lv_qid, lv_en = _space_qid(r.get("launch_vehicle")), _space_lbl(r, "launch_vehicle", "en")
        op_qid, op_en, op_ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([seed_qid, seed_en, lv_qid, lv_en, op_qid, op_en, y1, y2]): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P375 ?launch_vehicle .", "?item wdt:P375 ?launch_vehicle .", f"?item wdt:P137 wd:{op_qid} .", "FILTER(?item != ?seed) .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата, запущенных той же ракетой-носителем, что и «{seed_ru}», оператором которых является «{op_ru}» и которые были запущены в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft launched by the same launch vehicle as {seed_en}, operated by {op_en}, and launched between {y1} and {y2}.",
            "constraints": {"same_launch_vehicle_as": seed_en, "operator": op_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"seed_qid": seed_qid, "launch_vehicle_qid": lv_qid, "launch_vehicle_label_en": lv_en, "operator_qid": op_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_same_launch_vehicle_as_seed_operator_launch_period", "same_launch_vehicle_seed_operator_period", sparql, make_same_lv_op_period)

    # program + type + launch vehicle (direct 3-hop-ish constraints)
    where = f"""
      ?item wdt:P361 ?program ; wdt:P31/wdt:P279* ?type ; wdt:P375 ?launch_vehicle .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?type', 'type')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?program ?programLabelEn ?programLabelRu ?type ?typeLabelEn ?typeLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?program ?programLabelEn ?programLabelRu ?type ?typeLabelEn ?typeLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_program_type_lv(r):
        p_qid, p_en, p_ru = _space_qid(r.get("program")), _space_lbl(r, "program", "en"), _space_lbl(r, "program", "ru")
        t_qid, t_en, t_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        lv_qid, lv_en, lv_ru = _space_qid(r.get("launch_vehicle")), _space_lbl(r, "launch_vehicle", "en"), _space_lbl(r, "launch_vehicle", "ru")
        if not all([p_qid, p_en, t_qid, t_en, lv_qid, lv_en]): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P31/wdt:P279* wd:{t_qid} .", f"?item wdt:P375 wd:{lv_qid} ."],
            "query_text_ru": f"Назови 3 космических аппарата из программы/серии «{p_ru}», типа «{t_ru}» и запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 3 spacecraft that are part of the {p_en} program/series, are of type {t_en}, and were launched by {lv_en}.",
            "constraints": {"program_or_series": p_en, "type": t_en, "launch_vehicle": lv_en},
            "bridge_meta": {"program_qid": p_qid, "type_qid": t_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_program_type_launch_vehicle", "program_type_launch_vehicle", sparql, make_program_type_lv)

    records = [r for r in records if "candidate_error" not in r]
    records.sort(key=_space_candidate_sort_key)
    return records[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL[complexity]]


def build_spacecraft_candidates_l5() -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    complexity = "L5"
    years = _space_year_values("narrow")

    # same program as seed + launch vehicle + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P361 ?program .
      ?item wdt:P361 ?program ; wdt:P375 ?launch_vehicle ; wdt:P619 ?launch_date .
      FILTER(?item != ?seed)
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?program ?programLabelEn ?programLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2", where, "?seed ?seedLabelEn ?seedLabelRu ?program ?programLabelEn ?programLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2")
    def make_same_prog_lv_period(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get("seed")), _space_lbl(r, "seed", "en"), _space_lbl(r, "seed", "ru")
        p_qid, p_en = _space_qid(r.get("program")), _space_lbl(r, "program", "en")
        lv_qid, lv_en, lv_ru = _space_qid(r.get("launch_vehicle")), _space_lbl(r, "launch_vehicle", "en"), _space_lbl(r, "launch_vehicle", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([seed_qid, seed_en, p_qid, p_en, lv_qid, lv_en, y1, y2]): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P361 ?program .", "?item wdt:P361 ?program .", f"?item wdt:P375 wd:{lv_qid} .", "FILTER(?item != ?seed) .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата из той же программы/серии, что и «{seed_ru}», запущенных ракетой-носителем «{lv_ru}» в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft from the same program/series as {seed_en}, launched by {lv_en}, and launched between {y1} and {y2}.",
            "constraints": {"same_program_as": seed_en, "launch_vehicle": lv_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"seed_qid": seed_qid, "program_qid": p_qid, "program_label_en": p_en, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_same_program_as_seed_launch_vehicle_launch_period", "same_program_seed_launch_vehicle_period", sparql, make_same_prog_lv_period)

    # same operator as seed + manufacturer + type + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P137 ?operator .
      ?item wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P31/wdt:P279* ?type ; wdt:P619 ?launch_date .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      FILTER(?item != ?seed)
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?type', 'type')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?type ?typeLabelEn ?typeLabelRu ?y1 ?y2", where, "?seed ?seedLabelEn ?seedLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?type ?typeLabelEn ?typeLabelRu ?y1 ?y2")
    def make_same_op_manu_type_period(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get("seed")), _space_lbl(r, "seed", "en"), _space_lbl(r, "seed", "ru")
        op_qid, op_en = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en")
        manu_qid, manu_en, manu_ru = _space_qid(r.get("manufacturer")), _space_lbl(r, "manufacturer", "en"), _space_lbl(r, "manufacturer", "ru")
        type_qid, type_en, type_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([seed_qid, seed_en, op_qid, op_en, manu_qid, manu_en, type_qid, type_en, y1, y2]): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P137 ?operator .", "?item wdt:P137 ?operator .", f"?item wdt:P176 wd:{manu_qid} .", f"?item wdt:P31/wdt:P279* wd:{type_qid} .", "FILTER(?item != ?seed) .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата с тем же оператором, что и «{seed_ru}», произведённых организацией «{manu_ru}», типа «{type_ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft with the same operator as {seed_en}, manufactured by {manu_en}, of type {type_en}, and launched between {y1} and {y2}.",
            "constraints": {"same_operator_as": seed_en, "manufacturer": manu_en, "type": type_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"seed_qid": seed_qid, "operator_qid": op_qid, "manufacturer_qid": manu_qid, "type_qid": type_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_same_operator_as_seed_manufacturer_type_launch_period", "same_operator_seed_manufacturer_type_period", sparql, make_same_op_manu_type_period)

    # same launch vehicle as seed + operator + type + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P375 ?launch_vehicle .
      ?item wdt:P375 ?launch_vehicle ; wdt:P137 ?operator ; wdt:P31/wdt:P279* ?type ; wdt:P619 ?launch_date .
      ?type wdt:P279* wd:{Q_SPACECRAFT} .
      FILTER(?type != wd:{Q_SPACECRAFT})
      FILTER(?item != ?seed)
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?type', 'type')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?type ?typeLabelEn ?typeLabelRu ?y1 ?y2", where, "?seed ?seedLabelEn ?seedLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?type ?typeLabelEn ?typeLabelRu ?y1 ?y2")
    def make_same_lv_op_type_period(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get("seed")), _space_lbl(r, "seed", "en"), _space_lbl(r, "seed", "ru")
        lv_qid, lv_en = _space_qid(r.get("launch_vehicle")), _space_lbl(r, "launch_vehicle", "en")
        op_qid, op_en, op_ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        type_qid, type_en, type_ru = _space_qid(r.get("type")), _space_lbl(r, "type", "en"), _space_lbl(r, "type", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([seed_qid, seed_en, lv_qid, lv_en, op_qid, op_en, type_qid, type_en, y1, y2]): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P375 ?launch_vehicle .", "?item wdt:P375 ?launch_vehicle .", f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P31/wdt:P279* wd:{type_qid} .", "FILTER(?item != ?seed) .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата, запущенных той же ракетой-носителем, что и «{seed_ru}», оператором которых является «{op_ru}», типа «{type_ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft launched by the same launch vehicle as {seed_en}, operated by {op_en}, of type {type_en}, and launched between {y1} and {y2}.",
            "constraints": {"same_launch_vehicle_as": seed_en, "operator": op_en, "type": type_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"seed_qid": seed_qid, "launch_vehicle_qid": lv_qid, "operator_qid": op_qid, "type_qid": type_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_same_launch_vehicle_as_seed_operator_type_launch_period", "same_launch_vehicle_seed_operator_type_period", sparql, make_same_lv_op_type_period)

    # same country of origin as seed + operator + manufacturer + launch period.
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P495 ?country .
      ?item wdt:P495 ?country ; wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P619 ?launch_date .
      FILTER(?item != ?seed)
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?country', 'country')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?country ?countryLabelEn ?countryLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2", where, "?seed ?seedLabelEn ?seedLabelRu ?country ?countryLabelEn ?countryLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2")
    def make_same_country_op_manu_period(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get("seed")), _space_lbl(r, "seed", "en"), _space_lbl(r, "seed", "ru")
        c_qid, c_en = _space_qid(r.get("country")), _space_lbl(r, "country", "en")
        op_qid, op_en, op_ru = _space_qid(r.get("operator")), _space_lbl(r, "operator", "en"), _space_lbl(r, "operator", "ru")
        manu_qid, manu_en, manu_ru = _space_qid(r.get("manufacturer")), _space_lbl(r, "manufacturer", "en"), _space_lbl(r, "manufacturer", "ru")
        y1, y2 = _space_int(r.get("y1")), _space_int(r.get("y2"))
        if not all([seed_qid, seed_en, c_qid, c_en, op_qid, op_en, manu_qid, manu_en, y1, y2]): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P495 ?country .", "?item wdt:P495 ?country .", f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P176 wd:{manu_qid} .", "FILTER(?item != ?seed) .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата той же страны происхождения, что и «{seed_ru}», оператором которых является «{op_ru}», произведённых организацией «{manu_ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft with the same country of origin as {seed_en}, operated by {op_en}, manufactured by {manu_en}, and launched between {y1} and {y2}.",
            "constraints": {"same_country_of_origin_as": seed_en, "operator": op_en, "manufacturer": manu_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"seed_qid": seed_qid, "country_qid": c_qid, "country_label_en": c_en, "operator_qid": op_qid, "manufacturer_qid": manu_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_same_country_as_seed_operator_manufacturer_launch_period", "same_country_seed_operator_manufacturer_period", sparql, make_same_country_op_manu_period)

    records = [r for r in records if "candidate_error" not in r]
    records.sort(key=_space_candidate_sort_key)
    return records[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL[complexity]]


SPACECRAFT_CANDIDATE_BUILDERS = {
    "L1": build_spacecraft_candidates_l1,
    "L2": build_spacecraft_candidates_l2,
    "L3": build_spacecraft_candidates_l3,
    "L4": build_spacecraft_candidates_l4,
    "L5": build_spacecraft_candidates_l5,
}


def build_spacecraft_candidate_queue(complexity: str) -> List[Dict[str, Any]]:
    print(f"building spacecraft {complexity} candidate queue...")
    t0 = time.time()
    cands = SPACECRAFT_CANDIDATE_BUILDERS[complexity]()
    # Deduplicate exact constraints from candidate queue itself.
    seen = set()
    out = []
    for c in cands:
        sig = c.get("template_id", "") + "|" + _space_sig_constraints(_space_clean_constraints(c.get("constraints", {})))
        if sig in seen:
            continue
        seen.add(sig)
        out.append(c)
    rng = random.Random(SEED + sum(map(ord, complexity)))
    rng.shuffle(out)
    out.sort(key=_space_candidate_sort_key)
    print(f"spacecraft {complexity}: {len(out)} candidates built in {time.time()-t0:.1f}s")
    return out


# ============================================================
# JSONL I/O, validation, and incremental generation
# ============================================================

def read_existing_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception:
                pass
    return records


def append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()


def write_json(path: Path, data: Any) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def _space_level_next_idx(existing: List[Dict[str, Any]], level: str) -> int:
    vals = []
    pat = re.compile(rf"^spacecraft_{level.lower()}_(\d+)$")
    for r in existing:
        m = pat.match(str(r.get("id", "")))
        if m:
            vals.append(int(m.group(1)))
    return max(vals or [0]) + 1


def _space_record_quality_problems(r: Dict[str, Any]) -> List[str]:
    problems = []
    constraints = r.get("constraints") or {}
    lv = r.get("local_validator") or {}
    meta = r.get("gold_collection_meta") or {}
    if r.get("domain") != "spacecraft":
        problems.append("bad_domain")
    if r.get("complexity") not in SPACECRAFT_TARGET_PER_LEVEL:
        problems.append("bad_complexity")
    if len(r.get("gold_answer_qids") or []) < int(r.get("requested_count") or 0):
        problems.append("gold_lt_requested")
    if len(set(r.get("gold_answer_qids") or [])) != len(r.get("gold_answer_qids") or []):
        problems.append("duplicate_gold_qids")
    if len(r.get("gold_answer_qids") or []) != len(r.get("gold_answer_labels_ru") or []):
        problems.append("qid_ru_label_len_mismatch")
    if len(r.get("gold_answer_qids") or []) != len(r.get("gold_answer_labels_en") or []):
        problems.append("qid_en_label_len_mismatch")
    if lv.get("filters") != constraints:
        problems.append("constraints_local_validator_mismatch")
    if meta.get("constraints") != constraints:
        problems.append("constraints_meta_mismatch")
    dirty = json.dumps(constraints, ensure_ascii=False)
    if re.search(r"\bQ\d+\b|\bP\d+\b|wd:", dirty):
        problems.append("dirty_constraints")
    if "wdt:P31/wdt:P279* wd:" not in str(r.get("sparql_query", "")):
        problems.append("missing_spacecraft_class_filter")
    if r.get("gold_truncated") or meta.get("gold_may_be_incomplete_due_to_wdqs_limit") or meta.get("gold_truncated_by_local_limit"):
        problems.append("gold_truncated_or_incomplete")
    return problems


def _space_should_accept(record: Dict[str, Any], existing: List[Dict[str, Any]], counters: Dict[str, Counter]) -> Tuple[bool, str]:
    level = record["complexity"]
    sig = _space_sig_constraints(record["constraints"])
    query_ru = record.get("query_text_ru", "")
    qids = _space_gold_set(record)
    template_id = record.get("template_id", "")
    primary = _space_primary_key(record)

    if _space_record_quality_problems(record):
        return False, "quality_problems:" + ",".join(_space_record_quality_problems(record))
    if counters["template_level"][(level, template_id)] >= SPACECRAFT_MAX_SAME_TEMPLATE_PER_LEVEL[level]:
        return False, "template_cap"
    if counters["primary_level"][(level, primary)] >= SPACECRAFT_MAX_SAME_PRIMARY_PER_LEVEL[level]:
        return False, "primary_constraint_cap"
    for old in existing:
        if old.get("domain") != "spacecraft":
            continue
        if old.get("query_text_ru") == query_ru:
            return False, "duplicate_query"
        if _space_sig_constraints(old.get("constraints", {})) == sig:
            return False, "duplicate_constraints"
        old_qids = _space_gold_set(old)
        if not old_qids or not qids:
            continue
        j = _space_jaccard(qids, old_qids)
        cont = _space_containment(qids, old_qids)
        if j >= SPACECRAFT_GOLD_JACCARD_THRESHOLD:
            return False, f"gold_jaccard:{j:.3f}"
        if old.get("complexity") == level and old.get("template_family") == record.get("template_family") and cont >= SPACECRAFT_GOLD_CONTAINMENT_THRESHOLD:
            return False, f"gold_containment:{cont:.3f}"
    return True, "ok"


def generate_spacecraft_dataset() -> List[Dict[str, Any]]:
    existing = read_existing_jsonl(SPACECRAFT_OUTPUT_PATH)
    print("existing output records:", len(existing))
    print("existing counts:", dict(Counter(r.get("complexity") for r in existing)))
    skipped: List[Dict[str, Any]] = []

    counters = {
        "template_level": Counter((r.get("complexity"), r.get("template_id")) for r in existing),
        "primary_level": Counter((r.get("complexity"), _space_primary_key(r)) for r in existing),
    }
    existing_constraint_sigs = {_space_sig_constraints(r.get("constraints", {})) for r in existing if r.get("domain") == "spacecraft"}

    for level in ["L1", "L2", "L3", "L4", "L5"]:
        current = [r for r in existing if r.get("complexity") == level]
        need = max(0, SPACECRAFT_TARGET_PER_LEVEL[level] - len(current))
        if need <= 0:
            print(f"SKIP spacecraft:{level} already has {len(current)}/{SPACECRAFT_TARGET_PER_LEVEL[level]}")
            continue

        queue = build_spacecraft_candidate_queue(level)
        print(f"spacecraft:{level} target need: {need}; candidate queue: {len(queue)}")
        idx = _space_level_next_idx(existing, level)
        accepted_this_level = 0
        tried = 0

        for cand in queue:
            if accepted_this_level >= need:
                break
            tried += 1
            pre_ok, pre_reason = _space_should_try_candidate(cand, level, existing_constraint_sigs, counters)
            if not pre_ok:
                skipped.append({
                    "complexity": level,
                    "template_id": cand.get("template_id"),
                    "expected_count": cand.get("expected_count"),
                    "reason": pre_reason,
                    "constraints": _space_clean_constraints(cand.get("constraints", {})),
                })
                if tried % 200 == 0:
                    print(f"... spacecraft:{level} tried={tried}, accepted={accepted_this_level}/{need}, last_skip={pre_reason}")
                continue
            rec = _space_make_record(cand, idx)
            if rec is None:
                skipped.append({
                    "complexity": level,
                    "template_id": cand.get("template_id"),
                    "expected_count": cand.get("expected_count"),
                    "reason": cand.get("skip_reason", "make_record_failed"),
                    "constraints": _space_clean_constraints(cand.get("constraints", {})),
                })
                continue
            ok, reason = _space_should_accept(rec, existing, counters)
            if not ok:
                skipped.append({
                    "complexity": level,
                    "template_id": rec.get("template_id"),
                    "gold_count": len(rec.get("gold_answer_qids") or []),
                    "reason": reason,
                    "constraints": rec.get("constraints"),
                })
                continue

            append_jsonl(SPACECRAFT_OUTPUT_PATH, rec)
            existing.append(rec)
            counters["template_level"][(level, rec.get("template_id"))] += 1
            counters["primary_level"][(level, _space_primary_key(rec))] += 1
            existing_constraint_sigs.add(_space_sig_constraints(rec.get("constraints", {})))
            accepted_this_level += 1
            idx += 1
            print(
                f"OK spacecraft:{level} {accepted_this_level}/{need}; total={len(existing)}; "
                f"gold={len(rec['gold_answer_qids'])}; tpl={rec.get('template_id')}"
            )
            write_json(SPACECRAFT_CHECKPOINT_PATH, {
                "patch_version": SPACECRAFT_PATCH_VERSION,
                "counts": dict(Counter(r.get("complexity") for r in existing)),
                "total": len(existing),
                "last_id": rec.get("id"),
                "updated_at": utc_now_z(),
            })

        if accepted_this_level < need:
            print(f"WARNING: {level} target not reached: accepted {accepted_this_level}/{need}; tried={tried}; candidates={len(queue)}")

    audit = {
        "patch_version": SPACECRAFT_PATCH_VERSION,
        "output": str(SPACECRAFT_OUTPUT_PATH),
        "total": len(existing),
        "counts": dict(Counter(r.get("complexity") for r in existing)),
        "template_counts": dict(Counter(r.get("template_id") for r in existing)),
        "template_family_counts": dict(Counter(r.get("template_family") for r in existing)),
        "skipped_count": len(skipped),
        "skipped_reason_counts": dict(Counter(s.get("reason") for s in skipped)),
        "skipped_sample": skipped[:150],
        "validation": spacecraft_validate_records(existing),
        "updated_at": utc_now_z(),
    }
    write_json(SPACECRAFT_AUDIT_PATH, audit)
    print("final counts:", audit["counts"])
    print("skipped:", len(skipped))
    return existing


def spacecraft_validate_records(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    ids = [r.get("id") for r in records]
    out = {
        "total": len(records),
        "counts": dict(Counter(r.get("complexity") for r in records)),
        "duplicate_ids": [x for x, c in Counter(ids).items() if c > 1],
        "duplicate_query_text_ru": sum(1 for _, c in Counter(r.get("query_text_ru") for r in records).items() if c > 1),
        "duplicate_constraints": sum(1 for _, c in Counter(_space_sig_constraints(r.get("constraints", {})) for r in records).items() if c > 1),
        "dirty_constraints": [],
        "gold_lt_requested": [],
        "gold_duplicate_qids": [],
        "label_length_mismatch": [],
        "constraints_local_validator_mismatch": [],
        "constraints_meta_mismatch": [],
        "missing_class_filter": [],
        "gold_truncated_or_incomplete": [],
        "schema_errors": [],
    }
    for r in records:
        rid = r.get("id")
        probs = _space_record_quality_problems(r)
        if "dirty_constraints" in probs: out["dirty_constraints"].append(rid)
        if "gold_lt_requested" in probs: out["gold_lt_requested"].append(rid)
        if "duplicate_gold_qids" in probs: out["gold_duplicate_qids"].append(rid)
        if "qid_ru_label_len_mismatch" in probs or "qid_en_label_len_mismatch" in probs: out["label_length_mismatch"].append(rid)
        if "constraints_local_validator_mismatch" in probs: out["constraints_local_validator_mismatch"].append(rid)
        if "constraints_meta_mismatch" in probs: out["constraints_meta_mismatch"].append(rid)
        if "missing_spacecraft_class_filter" in probs: out["missing_class_filter"].append(rid)
        if "gold_truncated_or_incomplete" in probs: out["gold_truncated_or_incomplete"].append(rid)
        try:
            if "validate_record_schema" in globals():
                validate_record_schema(r)
        except Exception as e:
            out["schema_errors"].append({"id": rid, "error": str(e)})
    return out




# ============================================================
# v47 overrides: high-recall L2/L4/L5 candidate queues
# ============================================================
# Why this exists: v46 used overly strict L4/L5 seed+type+period candidate queries,
# so some levels built almost no candidates. v47 keeps the complete-gold checks,
# but broadens candidate discovery with cheaper high-recall intersections.

SPACECRAFT_PATCH_VERSION = "v47_recall_multilevel"
SPACECRAFT_TARGET_PER_LEVEL.update({"L1": 15, "L2": 22, "L3": 25, "L4": 30, "L5": 28})
SPACECRAFT_REQUESTED_COUNT.update({"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3})
SPACECRAFT_ACCEPT_MIN_GOLD.update({"L1": 7, "L2": 5, "L3": 5, "L4": 3, "L5": 3})
SPACECRAFT_ACCEPT_MAX_GOLD.update({"L1": 90, "L2": 110, "L3": 90, "L4": 80, "L5": 70})
SPACECRAFT_CANDIDATE_LIMIT_PER_TEMPLATE.update({"L2": 700, "L3": 700, "L4": 900, "L5": 1100})
SPACECRAFT_MAX_CANDIDATES_PER_LEVEL.update({"L2": 3500, "L3": 3600, "L4": 5000, "L5": 6000})
SPACECRAFT_MAX_SAME_TEMPLATE_PER_LEVEL.update({"L1": 5, "L2": 8, "L3": 9, "L4": 11, "L5": 12})
SPACECRAFT_MAX_SAME_PRIMARY_PER_LEVEL.update({"L1": 5, "L2": 7, "L3": 8, "L4": 10, "L5": 10})
SPACECRAFT_GOLD_JACCARD_THRESHOLD = 0.93
SPACECRAFT_GOLD_CONTAINMENT_THRESHOLD = 0.98

try:
    wd.timeout = min(max(int(getattr(wd, "timeout", 12)), 8), 14)
    wd.max_retries = 1
except Exception:
    pass


def _space_qid_required(*vals: Any) -> bool:
    return all(v is not None and v != "" for v in vals)


def _space_add_l2_high_recall(records: List[Dict[str, Any]]) -> None:
    """Extra L2 templates that avoid the sparse type path and generate many viable rows."""
    complexity = "L2"

    # operator + launch vehicle
    where = f"""
      ?item wdt:P137 ?operator ; wdt:P375 ?launch_vehicle .
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?operator ?operatorLabelEn ?operatorLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_op_lv(r):
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        if not _space_qid_required(op_qid, op_en, lv_qid, lv_en): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P375 wd:{lv_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов, оператором которых является «{op_ru}» и которые были запущены ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 5 spacecraft operated by {op_en} and launched by {lv_en}.",
            "constraints": {"operator": op_en, "launch_vehicle": lv_en},
            "bridge_meta": {"operator_qid": op_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_operator_launch_vehicle", "operator_launch_vehicle", sparql, make_op_lv)

    # manufacturer + launch vehicle
    where = f"""
      ?item wdt:P176 ?manufacturer ; wdt:P375 ?launch_vehicle .
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_manu_lv(r):
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_lbl(r, 'manufacturer', 'en'), _space_lbl(r, 'manufacturer', 'ru')
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        if not _space_qid_required(m_qid, m_en, lv_qid, lv_en): return None
        return {
            "where_lines": [f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P375 wd:{lv_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов, произведённых организацией «{m_ru}» и запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 5 spacecraft manufactured by {m_en} and launched by {lv_en}.",
            "constraints": {"manufacturer": m_en, "launch_vehicle": lv_en},
            "bridge_meta": {"manufacturer_qid": m_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_manufacturer_launch_vehicle", "manufacturer_launch_vehicle", sparql, make_manu_lv)

    # operator + country of origin
    where = f"""
      ?item wdt:P137 ?operator ; wdt:P495 ?country .
      ?country wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?country', 'country')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?country ?countryLabelEn ?countryLabelRu", where, "?operator ?operatorLabelEn ?operatorLabelRu ?country ?countryLabelEn ?countryLabelRu")
    def make_op_country(r):
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        c_qid, c_en, c_ru = _space_qid(r.get('country')), _space_lbl(r, 'country', 'en'), _space_lbl(r, 'country', 'ru')
        if not _space_qid_required(op_qid, op_en, c_qid, c_en): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P495 wd:{c_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов страны происхождения «{c_ru}», оператором которых является «{op_ru}».",
            "query_text_en": f"Name 5 spacecraft whose country of origin is {c_en} and that are operated by {op_en}.",
            "constraints": {"country_of_origin": c_en, "operator": op_en},
            "bridge_meta": {"operator_qid": op_qid, "country_qid": c_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_country_operator", "country_operator", sparql, make_op_country)

    # manufacturer + country of origin
    where = f"""
      ?item wdt:P176 ?manufacturer ; wdt:P495 ?country .
      ?country wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?country', 'country')}
    """
    sparql = _space_group_query(complexity, "?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?country ?countryLabelEn ?countryLabelRu", where, "?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?country ?countryLabelEn ?countryLabelRu")
    def make_manu_country(r):
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_lbl(r, 'manufacturer', 'en'), _space_lbl(r, 'manufacturer', 'ru')
        c_qid, c_en, c_ru = _space_qid(r.get('country')), _space_lbl(r, 'country', 'en'), _space_lbl(r, 'country', 'ru')
        if not _space_qid_required(m_qid, m_en, c_qid, c_en): return None
        return {
            "where_lines": [f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P495 wd:{c_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов страны происхождения «{c_ru}», произведённых организацией «{m_ru}».",
            "query_text_en": f"Name 5 spacecraft whose country of origin is {c_en} and that were manufactured by {m_en}.",
            "constraints": {"country_of_origin": c_en, "manufacturer": m_en},
            "bridge_meta": {"manufacturer_qid": m_qid, "country_qid": c_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_country_manufacturer", "country_manufacturer", sparql, make_manu_country)

    # program/series + operator
    where = f"""
      ?item wdt:P361 ?program ; wdt:P137 ?operator .
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?operator', 'operator')}
    """
    sparql = _space_group_query(complexity, "?program ?programLabelEn ?programLabelRu ?operator ?operatorLabelEn ?operatorLabelRu", where, "?program ?programLabelEn ?programLabelRu ?operator ?operatorLabelEn ?operatorLabelRu")
    def make_program_op(r):
        p_qid, p_en, p_ru = _space_qid(r.get('program')), _space_lbl(r, 'program', 'en'), _space_lbl(r, 'program', 'ru')
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        if not _space_qid_required(p_qid, p_en, op_qid, op_en): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P137 wd:{op_qid} ."],
            "query_text_ru": f"Назови 5 космических аппаратов из программы/серии «{p_ru}», оператором которых является «{op_ru}».",
            "query_text_en": f"Name 5 spacecraft that are part of {p_en} and are operated by {op_en}.",
            "constraints": {"program_or_series": p_en, "operator": op_en},
            "bridge_meta": {"program_qid": p_qid, "operator_qid": op_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l2_program_operator", "program_operator", sparql, make_program_op)


def build_spacecraft_candidates_l2_v47() -> List[Dict[str, Any]]:
    records = build_spacecraft_candidates_l2()
    _space_add_l2_high_recall(records)
    records = [r for r in records if "candidate_error" not in r]
    return _space_interleave_candidates(records)[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL["L2"]]


def build_spacecraft_candidates_l4_v47() -> List[Dict[str, Any]]:
    """L4: 3-4 concrete constraints, mostly direct intersections; robust and fast."""
    records: List[Dict[str, Any]] = []
    complexity = "L4"
    years = _space_year_values("mid")

    # operator + manufacturer + launch vehicle
    where = f"""
      ?item wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P375 ?launch_vehicle .
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_op_manu_lv(r):
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_lbl(r, 'manufacturer', 'en'), _space_lbl(r, 'manufacturer', 'ru')
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        if not _space_qid_required(op_qid, op_en, m_qid, m_en, lv_qid, lv_en): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P375 wd:{lv_qid} ."],
            "query_text_ru": f"Назови 3 космических аппарата, оператором которых является «{op_ru}», произведённых организацией «{m_ru}» и запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 3 spacecraft operated by {op_en}, manufactured by {m_en}, and launched by {lv_en}.",
            "constraints": {"operator": op_en, "manufacturer": m_en, "launch_vehicle": lv_en},
            "bridge_meta": {"operator_qid": op_qid, "manufacturer_qid": m_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_operator_manufacturer_launch_vehicle", "operator_manufacturer_launch_vehicle", sparql, make_op_manu_lv)

    # launch vehicle + operator + launch period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P375 ?launch_vehicle ; wdt:P137 ?operator ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
      {_space_label_lines('?operator', 'operator')}
    """
    sparql = _space_group_query(complexity, "?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?y1 ?y2", where, "?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?y1 ?y2")
    def make_lv_op_period(r):
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(lv_qid, lv_en, op_qid, op_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P375 wd:{lv_qid} .", f"?item wdt:P137 wd:{op_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата, запущенных ракетой-носителем «{lv_ru}», оператором которых является «{op_ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft launched by {lv_en}, operated by {op_en}, and launched between {y1} and {y2}.",
            "constraints": {"launch_vehicle": lv_en, "operator": op_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"launch_vehicle_qid": lv_qid, "operator_qid": op_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_launch_vehicle_operator_launch_period", "launch_vehicle_operator_period", sparql, make_lv_op_period)

    # program + launch vehicle + operator
    where = f"""
      ?item wdt:P361 ?program ; wdt:P375 ?launch_vehicle ; wdt:P137 ?operator .
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
      {_space_label_lines('?operator', 'operator')}
    """
    sparql = _space_group_query(complexity, "?program ?programLabelEn ?programLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu", where, "?program ?programLabelEn ?programLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?operator ?operatorLabelEn ?operatorLabelRu")
    def make_prog_lv_op(r):
        p_qid, p_en, p_ru = _space_qid(r.get('program')), _space_lbl(r, 'program', 'en'), _space_lbl(r, 'program', 'ru')
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        if not _space_qid_required(p_qid, p_en, lv_qid, lv_en, op_qid, op_en): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P375 wd:{lv_qid} .", f"?item wdt:P137 wd:{op_qid} ."],
            "query_text_ru": f"Назови 3 космических аппарата из программы/серии «{p_ru}», запущенных ракетой-носителем «{lv_ru}» и управляемых организацией «{op_ru}».",
            "query_text_en": f"Name 3 spacecraft that are part of {p_en}, were launched by {lv_en}, and are operated by {op_en}.",
            "constraints": {"program_or_series": p_en, "launch_vehicle": lv_en, "operator": op_en},
            "bridge_meta": {"program_qid": p_qid, "launch_vehicle_qid": lv_qid, "operator_qid": op_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_program_launch_vehicle_operator", "program_launch_vehicle_operator", sparql, make_prog_lv_op)

    # country + operator + manufacturer + period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P495 ?country ; wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P619 ?launch_date .
      ?country wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?country', 'country')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
    """
    sparql = _space_group_query(complexity, "?country ?countryLabelEn ?countryLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2", where, "?country ?countryLabelEn ?countryLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2")
    def make_country_op_manu_period(r):
        c_qid, c_en, c_ru = _space_qid(r.get('country')), _space_lbl(r, 'country', 'en'), _space_lbl(r, 'country', 'ru')
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_lbl(r, 'manufacturer', 'en'), _space_lbl(r, 'manufacturer', 'ru')
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(c_qid, c_en, op_qid, op_en, m_qid, m_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P495 wd:{c_qid} .", f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P176 wd:{m_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата страны происхождения «{c_ru}», оператором которых является «{op_ru}», произведённых организацией «{m_ru}» и запущенных в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft whose country of origin is {c_en}, operated by {op_en}, manufactured by {m_en}, and launched between {y1} and {y2}.",
            "constraints": {"country_of_origin": c_en, "operator": op_en, "manufacturer": m_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"country_qid": c_qid, "operator_qid": op_qid, "manufacturer_qid": m_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_country_operator_manufacturer_launch_period", "country_operator_manufacturer_period", sparql, make_country_op_manu_period)

    # include old v46 L4 as an extra source, but do not depend on it
    try:
        records.extend(build_spacecraft_candidates_l4())
    except Exception:
        pass
    records = [r for r in records if "candidate_error" not in r]
    return _space_interleave_candidates(records)[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL["L4"]]


def build_spacecraft_candidates_l5_v47() -> List[Dict[str, Any]]:
    """L5: bridge-style same-as constraints plus concrete intersections, but no over-strict type+period combo."""
    records: List[Dict[str, Any]] = []
    complexity = "L5"
    years = _space_year_values("mid")

    # same program as seed + launch vehicle (no period) -- high-recall true multihop
    where = f"""
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P361 ?program .
      ?item wdt:P361 ?program ; wdt:P375 ?launch_vehicle .
      FILTER(?item != ?seed)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?program ?programLabelEn ?programLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?seed ?seedLabelEn ?seedLabelRu ?program ?programLabelEn ?programLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_same_prog_lv(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get('seed')), _space_lbl(r, 'seed', 'en'), _space_lbl(r, 'seed', 'ru')
        p_qid, p_en = _space_qid(r.get('program')), _space_lbl(r, 'program', 'en')
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        if not _space_qid_required(seed_qid, seed_en, p_qid, p_en, lv_qid, lv_en): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P361 ?program .", "?item wdt:P361 ?program .", f"?item wdt:P375 wd:{lv_qid} .", "FILTER(?item != ?seed) ."],
            "query_text_ru": f"Назови 3 космических аппарата из той же программы/серии, что и «{seed_ru}», запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 3 spacecraft from the same program/series as {seed_en} that were launched by {lv_en}.",
            "constraints": {"same_program_as": seed_en, "launch_vehicle": lv_en},
            "bridge_meta": {"seed_qid": seed_qid, "program_qid": p_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_same_program_as_seed_launch_vehicle", "same_program_seed_launch_vehicle", sparql, make_same_prog_lv)

    # same operator as seed + manufacturer + launch vehicle
    where = f"""
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P137 ?operator .
      ?item wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P375 ?launch_vehicle .
      FILTER(?item != ?seed)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?seed ?seedLabelEn ?seedLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_same_op_manu_lv(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get('seed')), _space_lbl(r, 'seed', 'en'), _space_lbl(r, 'seed', 'ru')
        op_qid, op_en = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en')
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_lbl(r, 'manufacturer', 'en'), _space_lbl(r, 'manufacturer', 'ru')
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        if not _space_qid_required(seed_qid, seed_en, op_qid, op_en, m_qid, m_en, lv_qid, lv_en): return None
        return {
            "where_lines": [f"BIND(wd:{seed_qid} AS ?seed) .", "?seed wdt:P137 ?operator .", "?item wdt:P137 ?operator .", f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P375 wd:{lv_qid} .", "FILTER(?item != ?seed) ."],
            "query_text_ru": f"Назови 3 космических аппарата с тем же оператором, что и «{seed_ru}», произведённых организацией «{m_ru}» и запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 3 spacecraft with the same operator as {seed_en}, manufactured by {m_en}, and launched by {lv_en}.",
            "constraints": {"same_operator_as": seed_en, "manufacturer": m_en, "launch_vehicle": lv_en},
            "bridge_meta": {"seed_qid": seed_qid, "operator_qid": op_qid, "manufacturer_qid": m_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_same_operator_as_seed_manufacturer_launch_vehicle", "same_operator_seed_manufacturer_launch_vehicle", sparql, make_same_op_manu_lv)

    # direct 4-constraint fallback: operator + manufacturer + launch vehicle + period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P375 ?launch_vehicle ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2", where, "?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2")
    def make_op_manu_lv_period(r):
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_lbl(r, 'manufacturer', 'en'), _space_lbl(r, 'manufacturer', 'ru')
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(op_qid, op_en, m_qid, m_en, lv_qid, lv_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P375 wd:{lv_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата, оператором которых является «{op_ru}», произведённых организацией «{m_ru}», запущенных ракетой-носителем «{lv_ru}» в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft operated by {op_en}, manufactured by {m_en}, launched by {lv_en}, and launched between {y1} and {y2}.",
            "constraints": {"operator": op_en, "manufacturer": m_en, "launch_vehicle": lv_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"operator_qid": op_qid, "manufacturer_qid": m_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_operator_manufacturer_launch_vehicle_period", "operator_manufacturer_launch_vehicle_period", sparql, make_op_manu_lv_period)

    # direct program + operator + launch vehicle + period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P361 ?program ; wdt:P137 ?operator ; wdt:P375 ?launch_vehicle ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?program ?programLabelEn ?programLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2", where, "?program ?programLabelEn ?programLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2")
    def make_prog_op_lv_period(r):
        p_qid, p_en, p_ru = _space_qid(r.get('program')), _space_lbl(r, 'program', 'en'), _space_lbl(r, 'program', 'ru')
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_lbl(r, 'operator', 'en'), _space_lbl(r, 'operator', 'ru')
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_lbl(r, 'launch_vehicle', 'en'), _space_lbl(r, 'launch_vehicle', 'ru')
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(p_qid, p_en, op_qid, op_en, lv_qid, lv_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P375 wd:{lv_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата из программы/серии «{p_ru}», оператором которых является «{op_ru}», запущенных ракетой-носителем «{lv_ru}» в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft that are part of {p_en}, operated by {op_en}, launched by {lv_en}, and launched between {y1} and {y2}.",
            "constraints": {"program_or_series": p_en, "operator": op_en, "launch_vehicle": lv_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"program_qid": p_qid, "operator_qid": op_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_program_operator_launch_vehicle_period", "program_operator_launch_vehicle_period", sparql, make_prog_op_lv_period)

    # include old v46 L5 if it works on a machine/WDQS state
    try:
        records.extend(build_spacecraft_candidates_l5())
    except Exception:
        pass
    records = [r for r in records if "candidate_error" not in r]
    return _space_interleave_candidates(records)[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL["L5"]]


# Replace level builders after defining v47 high-recall versions.
SPACECRAFT_CANDIDATE_BUILDERS.update({
    "L2": build_spacecraft_candidates_l2_v47,
    "L4": build_spacecraft_candidates_l4_v47,
    "L5": build_spacecraft_candidates_l5_v47,
})


# ============================================================
# v48 overrides: quality fixes, stricter dedup, diverse L4/L5
# ============================================================
# Fixes found after validating v47 raw output:
# - L5 was too one-template-heavy;
# - several records had high gold overlap;
# - some RU/EN texts had duplicate quotes / "program program/series" / repeated "launched";
# - a few records had extra no-English-label matches in diagnostic counts;
# - operator==manufacturer and program-like operators made weak questions.

SPACECRAFT_PATCH_VERSION = "v48_quality_diverse_l5"

# Realistic 100–120 target. L5=20 is achievable with quality constraints; final curation can keep 100–110.
SPACECRAFT_TARGET_PER_LEVEL.update({"L1": 15, "L2": 22, "L3": 25, "L4": 28, "L5": 20})
SPACECRAFT_REQUESTED_COUNT.update({"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3})
SPACECRAFT_ACCEPT_MIN_GOLD.update({"L1": 7, "L2": 5, "L3": 5, "L4": 3, "L5": 3})
SPACECRAFT_ACCEPT_MAX_GOLD.update({"L1": 70, "L2": 70, "L3": 60, "L4": 45, "L5": 35})
SPACECRAFT_CANDIDATE_LIMIT_PER_TEMPLATE.update({"L2": 850, "L3": 850, "L4": 1000, "L5": 1300})
SPACECRAFT_MAX_CANDIDATES_PER_LEVEL.update({"L2": 4200, "L3": 4300, "L4": 6200, "L5": 7800})

# Stricter caps to prevent one-template L5 collapse.
SPACECRAFT_MAX_SAME_TEMPLATE_PER_LEVEL.update({"L1": 5, "L2": 7, "L3": 8, "L4": 7, "L5": 5})
SPACECRAFT_MAX_SAME_PRIMARY_PER_LEVEL.update({"L1": 4, "L2": 5, "L3": 6, "L4": 6, "L5": 6})
SPACECRAFT_GOLD_JACCARD_THRESHOLD = 0.88
SPACECRAFT_GOLD_CONTAINMENT_THRESHOLD = 0.94

# Keep single-candidate calls bounded.
SPACECRAFT_HARD_QUERY_TIMEOUT_SECONDS = 22
SPACECRAFT_AGG_QUERY_TIMEOUT_SECONDS = 35
try:
    wd.timeout = min(max(int(getattr(wd, "timeout", 12)), 8), 13)
    wd.max_retries = 1
except Exception:
    pass

_SPACECRAFT_PROGRAMLIKE_OPERATOR_RE = re.compile(r"(?i)\b(space program|space programme|program|programme|interkosmos)\b")
_SPACECRAFT_LOW_VALUE_ORG_RE = re.compile(r"(?i)\b(umbra|varda space industries)\b")


def _space_norm_label_for_text(x: Any) -> str:
    s = str(x or "").strip()
    s = re.sub(r"\s+", " ", s)
    # Do not nest Russian guillemets inside another pair of guillemets.
    s = s.replace("«", "\"").replace("»", "\"")
    s = re.sub(r'"+', '"', s)
    return s.strip(' "')


def _space_fix_ru_nested_quotes(text: str) -> str:
    s = str(text or "")
    old = None
    while old != s:
        old = s
        s = re.sub(r"«\s*«([^«»]+)»\s*»", r"«\1»", s)
        s = re.sub(r"«([^«»]*)«([^«»]+)»([^«»]*)»", lambda m: "«" + (m.group(1) + '"' + m.group(2) + '"' + m.group(3)).strip() + "»", s)
    return s


def _space_fix_query_text_ru(text: str) -> str:
    s = _space_fix_ru_nested_quotes(text)
    s = s.replace(" и запущенных в период ", " с датой запуска в период ")
    s = s.replace(" и которые были запущены в период ", " с датой запуска в период ")
    s = s.replace("которые которые", "которые")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _space_fix_query_text_en(text: str) -> str:
    s = str(text or "")
    s = re.sub(r"\bprogram program/series\b", "program", s)
    s = re.sub(r"\bprogramme program/series\b", "programme", s)
    s = s.replace("program/series program/series", "program/series")
    s = s.replace("part of the the ", "part of the ")
    # Remove repeated launched wording while preserving constraints.
    s = s.replace(", and launched between ", ", with launch dates between ")
    s = s.replace(", and were launched between ", ", with launch dates between ")
    s = re.sub(r"launched by ([^,]+), operated by ([^,]+), with launch dates between", r"launched by \1, operated by \2, with launch dates between", s)
    s = re.sub(r"launched by ([^,]+), manufactured by ([^,]+), with launch dates between", r"launched by \1, manufactured by \2, with launch dates between", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _space_sanitize_record_texts(rec: Dict[str, Any]) -> Dict[str, Any]:
    rec = dict(rec)
    rec["query_text_ru"] = _space_fix_query_text_ru(rec.get("query_text_ru", ""))
    rec["query_text_en"] = _space_fix_query_text_en(rec.get("query_text_en", ""))
    return rec


def _space_query_text_has_issue(rec: Dict[str, Any]) -> bool:
    ru = rec.get("query_text_ru", "")
    en = rec.get("query_text_en", "")
    bad_patterns = [
        "««", "»»", "program program/series", "programme program/series", "которые которые",
        ", and launched between", ", and were launched between",
    ]
    return any(p in ru or p in en for p in bad_patterns)


def _space_bad_constraints_for_quality(constraints: Dict[str, Any], level: str) -> Optional[str]:
    op = str(constraints.get("operator") or "").strip()
    manu = str(constraints.get("manufacturer") or "").strip()
    # operator==manufacturer is technically possible but usually too redundant for this benchmark.
    if op and manu and op.casefold() == manu.casefold():
        return "operator_equals_manufacturer"
    # Avoid cases where P137 is filled by a program/initiative rather than a real operator organization.
    if op and _SPACECRAFT_PROGRAMLIKE_OPERATOR_RE.search(op):
        return "program_like_operator"
    # Avoid low-value serial satellite sets in easier levels; they produce near-duplicate golds.
    if level in {"L1", "L2"} and (op and _SPACECRAFT_LOW_VALUE_ORG_RE.search(op) or manu and _SPACECRAFT_LOW_VALUE_ORG_RE.search(manu)):
        return "low_value_serial_operator_or_manufacturer"
    return None


# Save v47 implementations and wrap them.
_space_make_record_v47 = _space_make_record
_space_should_try_candidate_v47 = _space_should_try_candidate
_space_should_accept_v47 = _space_should_accept
_space_validate_records_v47 = spacecraft_validate_records


def _space_make_record(cand: Dict[str, Any], idx: int) -> Optional[Dict[str, Any]]:
    rec = _space_make_record_v47(cand, idx)
    if rec is None:
        return None
    rec = _space_sanitize_record_texts(rec)
    if _space_query_text_has_issue(rec):
        cand["skip_reason"] = "query_text_issue_after_sanitize"
        return None
    meta = rec.get("gold_collection_meta") or {}
    complete = meta.get("complete_count_with_en_label")
    diagnostic = meta.get("diagnostic_count_without_label_filter")
    # For maximum gold completeness, reject records where there are matching QIDs without English labels.
    # Otherwise the gold list is complete only within the English-label universe, not full WDQS.
    if diagnostic is None:
        cand["skip_reason"] = "diagnostic_count_missing"
        return None
    try:
        if int(diagnostic) > int(complete):
            cand["skip_reason"] = f"unlabeled_matching_items:{complete}_of_{diagnostic}"
            return None
    except Exception:
        pass
    return rec


def _space_should_try_candidate(cand: Dict[str, Any], level: str, existing_constraint_sigs: set, counters: Dict[str, Counter]) -> Tuple[bool, str]:
    constraints = _space_clean_constraints(cand.get("constraints", {}))
    bad = _space_bad_constraints_for_quality(constraints, level)
    if bad:
        return False, bad
    return _space_should_try_candidate_v47(cand, level, existing_constraint_sigs, counters)


def _space_should_accept(record: Dict[str, Any], existing: List[Dict[str, Any]], counters: Dict[str, Counter]) -> Tuple[bool, str]:
    bad = _space_bad_constraints_for_quality(record.get("constraints", {}), record.get("complexity", ""))
    if bad:
        return False, bad
    if _space_query_text_has_issue(record):
        return False, "query_text_issue"
    ok, reason = _space_should_accept_v47(record, existing, counters)
    if not ok:
        return ok, reason
    qids = _space_gold_set(record)
    for old in existing:
        if old.get("domain") != "spacecraft" or not _space_gold_set(old):
            continue
        # Stronger overlap control if records share a primary dimension or template family.
        same_axis = _space_primary_key(record) == _space_primary_key(old) or record.get("template_family") == old.get("template_family")
        j = _space_jaccard(qids, _space_gold_set(old))
        cont = _space_containment(qids, _space_gold_set(old))
        if same_axis and j >= 0.82:
            return False, f"v48_axis_gold_jaccard:{j:.3f}"
        if same_axis and cont >= 0.90:
            return False, f"v48_axis_gold_containment:{cont:.3f}"
    return True, "ok"


def spacecraft_validate_records(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    out = _space_validate_records_v47(records)
    out["query_text_issues"] = [r.get("id") for r in records if _space_query_text_has_issue(r)]
    out["operator_equals_manufacturer"] = [
        r.get("id") for r in records
        if str((r.get("constraints") or {}).get("operator") or "").casefold()
        and str((r.get("constraints") or {}).get("operator") or "").casefold() == str((r.get("constraints") or {}).get("manufacturer") or "").casefold()
    ]
    out["program_like_operator"] = [
        r.get("id") for r in records
        if _SPACECRAFT_PROGRAMLIKE_OPERATOR_RE.search(str((r.get("constraints") or {}).get("operator") or ""))
    ]
    out["diagnostic_extra_unlabeled"] = []
    out["diagnostic_missing"] = []
    for r in records:
        meta = r.get("gold_collection_meta") or {}
        complete = meta.get("complete_count_with_en_label")
        diagnostic = meta.get("diagnostic_count_without_label_filter")
        if diagnostic is None:
            out["diagnostic_missing"].append(r.get("id"))
            continue
        try:
            if int(diagnostic) > int(complete):
                out["diagnostic_extra_unlabeled"].append({"id": r.get("id"), "complete_en": complete, "total_no_label_filter": diagnostic})
        except Exception:
            pass
    overlap_pairs = []
    for i, a in enumerate(records):
        aq = _space_gold_set(a)
        if not aq: continue
        for b in records[i+1:]:
            bq = _space_gold_set(b)
            if not bq: continue
            j = _space_jaccard(aq, bq)
            cont = _space_containment(aq, bq)
            if j >= 0.82 or cont >= 0.92:
                overlap_pairs.append({"a": a.get("id"), "b": b.get("id"), "jaccard": round(j, 3), "containment": round(cont, 3)})
    out["high_gold_overlap_pairs"] = overlap_pairs[:100]
    out["high_gold_overlap_pair_count"] = len(overlap_pairs)
    return out


def _space_repair_existing_jsonl(path: Path = SPACECRAFT_OUTPUT_PATH) -> None:
    """Non-destructive text repair for resumed runs. Does not delete records."""
    if not path.exists():
        return
    records = read_existing_jsonl(path)
    if not records:
        return
    repaired = [_space_sanitize_record_texts(r) if r.get("domain") == "spacecraft" else r for r in records]
    if repaired != records:
        tmp = path.with_suffix(path.suffix + ".tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            for r in repaired:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        tmp.replace(path)
        print(f"repaired existing spacecraft query texts in {path}")


# Extra diverse L4/L5 templates. They are deliberately based on robust, well-populated properties:
# P361 program/series, P137 operator, P176 manufacturer, P375 launch vehicle, P495 country, P619 date.

def _space_add_l4_v48_extra(records: List[Dict[str, Any]]) -> None:
    complexity = "L4"
    years = _space_year_values("mid")

    # program + manufacturer + launch vehicle
    where = f"""
      ?item wdt:P361 ?program ; wdt:P176 ?manufacturer ; wdt:P375 ?launch_vehicle .
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?program ?programLabelEn ?programLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?program ?programLabelEn ?programLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_prog_manu_lv(r):
        p_qid, p_en, p_ru = _space_qid(r.get('program')), _space_norm_label_for_text(_space_lbl(r, 'program', 'en')), _space_norm_label_for_text(_space_lbl(r, 'program', 'ru'))
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'en')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'ru'))
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'en')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'ru'))
        if not _space_qid_required(p_qid, p_en, m_qid, m_en, lv_qid, lv_en): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P375 wd:{lv_qid} ."],
            "query_text_ru": f"Назови 3 космических аппарата из программы/серии «{p_ru}», произведённых организацией «{m_ru}» и запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 3 spacecraft that are part of {p_en}, manufactured by {m_en}, and launched by {lv_en}.",
            "constraints": {"program_or_series": p_en, "manufacturer": m_en, "launch_vehicle": lv_en},
            "bridge_meta": {"program_qid": p_qid, "manufacturer_qid": m_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_program_manufacturer_launch_vehicle", "program_manufacturer_launch_vehicle", sparql, make_prog_manu_lv)

    # country + operator + launch vehicle + period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P495 ?country ; wdt:P137 ?operator ; wdt:P375 ?launch_vehicle ; wdt:P619 ?launch_date .
      ?country wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?country', 'country')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
      FILTER(!CONTAINS(LCASE(STR(?operatorLabelEn)), "program"))
    """
    sparql = _space_group_query(complexity, "?country ?countryLabelEn ?countryLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2", where, "?country ?countryLabelEn ?countryLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2")
    def make_country_op_lv_period(r):
        c_qid, c_en, c_ru = _space_qid(r.get('country')), _space_norm_label_for_text(_space_lbl(r, 'country', 'en')), _space_norm_label_for_text(_space_lbl(r, 'country', 'ru'))
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_norm_label_for_text(_space_lbl(r, 'operator', 'en')), _space_norm_label_for_text(_space_lbl(r, 'operator', 'ru'))
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'en')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'ru'))
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(c_qid, c_en, op_qid, op_en, lv_qid, lv_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P495 wd:{c_qid} .", f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P375 wd:{lv_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата страны происхождения «{c_ru}», оператором которых является «{op_ru}», запущенных ракетой-носителем «{lv_ru}» с датой запуска в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft whose country of origin is {c_en}, operated by {op_en}, launched by {lv_en}, with launch dates between {y1} and {y2}.",
            "constraints": {"country_of_origin": c_en, "operator": op_en, "launch_vehicle": lv_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"country_qid": c_qid, "operator_qid": op_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l4_country_operator_launch_vehicle_period", "country_operator_launch_vehicle_period", sparql, make_country_op_lv_period)


def _space_add_l5_v48_extra(records: List[Dict[str, Any]]) -> None:
    complexity = "L5"
    years = _space_year_values("narrow")

    # program + manufacturer + launch vehicle + period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P361 ?program ; wdt:P176 ?manufacturer ; wdt:P375 ?launch_vehicle ; wdt:P619 ?launch_date .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?program ?programLabelEn ?programLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2", where, "?program ?programLabelEn ?programLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2")
    def make_prog_manu_lv_period(r):
        p_qid, p_en, p_ru = _space_qid(r.get('program')), _space_norm_label_for_text(_space_lbl(r, 'program', 'en')), _space_norm_label_for_text(_space_lbl(r, 'program', 'ru'))
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'en')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'ru'))
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'en')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'ru'))
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(p_qid, p_en, m_qid, m_en, lv_qid, lv_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P375 wd:{lv_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата из программы/серии «{p_ru}», произведённых организацией «{m_ru}», запущенных ракетой-носителем «{lv_ru}» с датой запуска в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft that are part of {p_en}, manufactured by {m_en}, launched by {lv_en}, with launch dates between {y1} and {y2}.",
            "constraints": {"program_or_series": p_en, "manufacturer": m_en, "launch_vehicle": lv_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"program_qid": p_qid, "manufacturer_qid": m_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_program_manufacturer_launch_vehicle_period", "program_manufacturer_launch_vehicle_period", sparql, make_prog_manu_lv_period)

    # program + operator + manufacturer + period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P361 ?program ; wdt:P137 ?operator ; wdt:P176 ?manufacturer ; wdt:P619 ?launch_date .
      FILTER(?operator != ?manufacturer)
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      FILTER(!CONTAINS(LCASE(STR(?operatorLabelEn)), "program"))
    """
    sparql = _space_group_query(complexity, "?program ?programLabelEn ?programLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2", where, "?program ?programLabelEn ?programLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?y1 ?y2")
    def make_prog_op_manu_period(r):
        p_qid, p_en, p_ru = _space_qid(r.get('program')), _space_norm_label_for_text(_space_lbl(r, 'program', 'en')), _space_norm_label_for_text(_space_lbl(r, 'program', 'ru'))
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_norm_label_for_text(_space_lbl(r, 'operator', 'en')), _space_norm_label_for_text(_space_lbl(r, 'operator', 'ru'))
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'en')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'ru'))
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(p_qid, p_en, op_qid, op_en, m_qid, m_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P176 wd:{m_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата из программы/серии «{p_ru}», оператором которых является «{op_ru}», произведённых организацией «{m_ru}» с датой запуска в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft that are part of {p_en}, operated by {op_en}, manufactured by {m_en}, with launch dates between {y1} and {y2}.",
            "constraints": {"program_or_series": p_en, "operator": op_en, "manufacturer": m_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"program_qid": p_qid, "operator_qid": op_qid, "manufacturer_qid": m_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_program_operator_manufacturer_period", "program_operator_manufacturer_period", sparql, make_prog_op_manu_period)

    # country + operator + launch vehicle + period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P495 ?country ; wdt:P137 ?operator ; wdt:P375 ?launch_vehicle ; wdt:P619 ?launch_date .
      ?country wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?country', 'country')}
      {_space_label_lines('?operator', 'operator')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
      FILTER(!CONTAINS(LCASE(STR(?operatorLabelEn)), "program"))
    """
    sparql = _space_group_query(complexity, "?country ?countryLabelEn ?countryLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2", where, "?country ?countryLabelEn ?countryLabelRu ?operator ?operatorLabelEn ?operatorLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2")
    def make_country_op_lv_period(r):
        c_qid, c_en, c_ru = _space_qid(r.get('country')), _space_norm_label_for_text(_space_lbl(r, 'country', 'en')), _space_norm_label_for_text(_space_lbl(r, 'country', 'ru'))
        op_qid, op_en, op_ru = _space_qid(r.get('operator')), _space_norm_label_for_text(_space_lbl(r, 'operator', 'en')), _space_norm_label_for_text(_space_lbl(r, 'operator', 'ru'))
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'en')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'ru'))
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(c_qid, c_en, op_qid, op_en, lv_qid, lv_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P495 wd:{c_qid} .", f"?item wdt:P137 wd:{op_qid} .", f"?item wdt:P375 wd:{lv_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата страны происхождения «{c_ru}», оператором которых является «{op_ru}», запущенных ракетой-носителем «{lv_ru}» с датой запуска в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft whose country of origin is {c_en}, operated by {op_en}, launched by {lv_en}, with launch dates between {y1} and {y2}.",
            "constraints": {"country_of_origin": c_en, "operator": op_en, "launch_vehicle": lv_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"country_qid": c_qid, "operator_qid": op_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_country_operator_launch_vehicle_period", "country_operator_launch_vehicle_period", sparql, make_country_op_lv_period)

    # country + manufacturer + launch vehicle + period
    where = f"""
      VALUES (?y1 ?y2) {{
{years}
      }}
      ?item wdt:P495 ?country ; wdt:P176 ?manufacturer ; wdt:P375 ?launch_vehicle ; wdt:P619 ?launch_date .
      ?country wdt:P31/wdt:P279* wd:{Q_COUNTRY} .
      BIND(YEAR(?launch_date) AS ?launch_year)
      FILTER(?launch_year >= ?y1 && ?launch_year <= ?y2)
      {_space_label_lines('?country', 'country')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?country ?countryLabelEn ?countryLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2", where, "?country ?countryLabelEn ?countryLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu ?y1 ?y2")
    def make_country_manu_lv_period(r):
        c_qid, c_en, c_ru = _space_qid(r.get('country')), _space_norm_label_for_text(_space_lbl(r, 'country', 'en')), _space_norm_label_for_text(_space_lbl(r, 'country', 'ru'))
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'en')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'ru'))
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'en')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'ru'))
        y1, y2 = _space_int(r.get('y1')), _space_int(r.get('y2'))
        if not _space_qid_required(c_qid, c_en, m_qid, m_en, lv_qid, lv_en, y1, y2): return None
        return {
            "where_lines": [f"?item wdt:P495 wd:{c_qid} .", f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P375 wd:{lv_qid} .", *_space_fixed_year_filter(y1, y2)],
            "query_text_ru": f"Назови 3 космических аппарата страны происхождения «{c_ru}», произведённых организацией «{m_ru}», запущенных ракетой-носителем «{lv_ru}» с датой запуска в период {y1}–{y2} годов.",
            "query_text_en": f"Name 3 spacecraft whose country of origin is {c_en}, manufactured by {m_en}, launched by {lv_en}, with launch dates between {y1} and {y2}.",
            "constraints": {"country_of_origin": c_en, "manufacturer": m_en, "launch_vehicle": lv_en, "launch_year_from": y1, "launch_year_to": y2},
            "bridge_meta": {"country_qid": c_qid, "manufacturer_qid": m_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_country_manufacturer_launch_vehicle_period", "country_manufacturer_launch_vehicle_period", sparql, make_country_manu_lv_period)

    # seed bridge: same program as seed + manufacturer + launch vehicle
    where = f"""
      ?seed wdt:P31/wdt:P279* wd:{Q_SPACECRAFT} ; wdt:P361 ?program .
      ?item wdt:P361 ?program ; wdt:P176 ?manufacturer ; wdt:P375 ?launch_vehicle .
      FILTER(?item != ?seed)
      {_space_label_lines('?seed', 'seed')}
      {_space_label_lines('?program', 'program')}
      {_space_label_lines('?manufacturer', 'manufacturer')}
      {_space_label_lines('?launch_vehicle', 'launch_vehicle')}
    """
    sparql = _space_group_query(complexity, "?seed ?seedLabelEn ?seedLabelRu ?program ?programLabelEn ?programLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu", where, "?seed ?seedLabelEn ?seedLabelRu ?program ?programLabelEn ?programLabelRu ?manufacturer ?manufacturerLabelEn ?manufacturerLabelRu ?launch_vehicle ?launch_vehicleLabelEn ?launch_vehicleLabelRu")
    def make_same_prog_seed_manu_lv(r):
        seed_qid, seed_en, seed_ru = _space_qid(r.get('seed')), _space_norm_label_for_text(_space_lbl(r, 'seed', 'en')), _space_norm_label_for_text(_space_lbl(r, 'seed', 'ru'))
        p_qid, p_en, p_ru = _space_qid(r.get('program')), _space_norm_label_for_text(_space_lbl(r, 'program', 'en')), _space_norm_label_for_text(_space_lbl(r, 'program', 'ru'))
        m_qid, m_en, m_ru = _space_qid(r.get('manufacturer')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'en')), _space_norm_label_for_text(_space_lbl(r, 'manufacturer', 'ru'))
        lv_qid, lv_en, lv_ru = _space_qid(r.get('launch_vehicle')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'en')), _space_norm_label_for_text(_space_lbl(r, 'launch_vehicle', 'ru'))
        if not _space_qid_required(seed_qid, seed_en, p_qid, p_en, m_qid, m_en, lv_qid, lv_en): return None
        return {
            "where_lines": [f"?item wdt:P361 wd:{p_qid} .", f"?item wdt:P176 wd:{m_qid} .", f"?item wdt:P375 wd:{lv_qid} .", f"FILTER(?item != wd:{seed_qid})"],
            "query_text_ru": f"Назови 3 космических аппарата из той же программы/серии, что и «{seed_ru}», произведённых организацией «{m_ru}» и запущенных ракетой-носителем «{lv_ru}».",
            "query_text_en": f"Name 3 spacecraft from the same program/series as {seed_en}, manufactured by {m_en}, and launched by {lv_en}.",
            "constraints": {"same_program_as": seed_en, "program_or_series": p_en, "manufacturer": m_en, "launch_vehicle": lv_en},
            "bridge_meta": {"seed_qid": seed_qid, "program_qid": p_qid, "manufacturer_qid": m_qid, "launch_vehicle_qid": lv_qid},
        }
    _space_add_template_rows(records, complexity, "spacecraft_l5_same_program_seed_manufacturer_launch_vehicle", "same_program_seed_manufacturer_launch_vehicle", sparql, make_same_prog_seed_manu_lv)


def build_spacecraft_candidates_l4_v48() -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    try:
        records.extend(build_spacecraft_candidates_l4_v47())
    except Exception:
        pass
    _space_add_l4_v48_extra(records)
    records = [r for r in records if "candidate_error" not in r]
    # Candidate-level clean-up before expensive gold SELECT.
    records = [r for r in records if not _space_bad_constraints_for_quality(_space_clean_constraints(r.get("constraints", {})), "L4")]
    return _space_interleave_candidates(records)[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL["L4"]]


def build_spacecraft_candidates_l5_v48() -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    # Add v48 diverse templates first, then keep v47 as fallback.
    _space_add_l5_v48_extra(records)
    try:
        records.extend(build_spacecraft_candidates_l5_v47())
    except Exception:
        pass
    records = [r for r in records if "candidate_error" not in r]
    records = [r for r in records if not _space_bad_constraints_for_quality(_space_clean_constraints(r.get("constraints", {})), "L5")]
    return _space_interleave_candidates(records)[:SPACECRAFT_MAX_CANDIDATES_PER_LEVEL["L5"]]


SPACECRAFT_CANDIDATE_BUILDERS.update({
    "L4": build_spacecraft_candidates_l4_v48,
    "L5": build_spacecraft_candidates_l5_v48,
})

# Repair text bugs if resuming from an already generated file. For the cleanest run, delete old spacecraft.jsonl first.
SPACECRAFT_REPAIR_EXISTING_OUTPUT_TEXT = True


RUN_SPACECRAFT_GENERATION = True
if RUN_SPACECRAFT_GENERATION:
    if globals().get("SPACECRAFT_REPAIR_EXISTING_OUTPUT_TEXT", False):
        _space_repair_existing_jsonl(SPACECRAFT_OUTPUT_PATH)
    spacecraft_records = generate_spacecraft_dataset()
else:
    spacecraft_records = read_existing_jsonl(SPACECRAFT_OUTPUT_PATH)
    print(json.dumps(spacecraft_validate_records(spacecraft_records), ensure_ascii=False, indent=2))


Spacecraft patch: v47_recall_multilevel
output: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/spacecraft1.jsonl
existing output records: 0
existing counts: {}
building spacecraft L1 candidate queue...
spacecraft L1: 320 candidates built in 22.4s
spacecraft:L1 target need: 15; candidate queue: 320
OK spacecraft:L1 1/15; total=1; gold=10; tpl=spacecraft_l1_launch_vehicle_launch_period
OK spacecraft:L1 2/15; total=2; gold=10; tpl=spacecraft_l1_launch_vehicle_launch_period
OK spacecraft:L1 3/15; total=3; gold=10; tpl=spacecraft_l1_launch_vehicle_launch_period
OK spacecraft:L1 4/15; total=4; gold=10; tpl=spacecraft_l1_launch_vehicle_launch_period
OK spacecraft:L1 5/15; total=5; gold=10; tpl=spacecraft_l1_launch_vehicle_launch_period
OK spacecraft:L1 6/15; total=6; gold=10; tpl=spacecraft_l1_manufacturer_launch_period
OK spacecraft:L1 7/15; total=7; gold=10; tpl=spacecraft_l1_manufacturer_launch_period
OK spacecraft:L1 8/15; total=8

In [3]:

# ============================================================
# Spacecraft preview / schema sanity check
# ============================================================
REPORT_PATH = SPACECRAFT_OUTPUT_PATH
if REPORT_PATH.exists():
    rows = read_existing_jsonl(REPORT_PATH)
    print("rows:", len(rows))
    print("counts:", dict(Counter(r.get("complexity") for r in rows)))
    print("template counts:", dict(Counter(r.get("template_id") for r in rows)))
    print("template family counts:", dict(Counter(r.get("template_family") for r in rows)))
    print("validation:")
    print(json.dumps(spacecraft_validate_records(rows), ensure_ascii=False, indent=2))
    for r in rows[:5]:
        print("\n", r.get("id"), r.get("complexity"), r.get("template_id"), "gold=", len(r.get("gold_answer_qids") or []))
        print("RU:", r.get("query_text_ru"))
        print("EN:", r.get("query_text_en"))
        print("constraints:", json.dumps(r.get("constraints", {}), ensure_ascii=False))
else:
    print("No spacecraft.jsonl yet. Run the generation cell first.")


rows: 102
counts: {'L1': 15, 'L2': 22, 'L3': 24, 'L4': 21, 'L5': 20}
template counts: {'spacecraft_l1_launch_vehicle_launch_period': 5, 'spacecraft_l1_manufacturer_launch_period': 5, 'spacecraft_l1_operator_launch_period': 5, 'spacecraft_l2_manufacturer_launch_vehicle': 7, 'spacecraft_l2_operator_launch_vehicle': 7, 'spacecraft_l2_operator_manufacturer': 5, 'spacecraft_l2_program_operator': 3, 'spacecraft_l3_launch_vehicle_operator_launch_period': 8, 'spacecraft_l3_operator_manufacturer_launch_period': 8, 'spacecraft_l3_program_launch_vehicle': 8, 'spacecraft_l4_program_manufacturer_launch_vehicle': 7, 'spacecraft_l4_program_launch_vehicle_operator': 7, 'spacecraft_l4_launch_vehicle_operator_launch_period': 7, 'spacecraft_l5_program_operator_manufacturer_period': 5, 'spacecraft_l5_operator_manufacturer_launch_vehicle_period': 5, 'spacecraft_l5_program_manufacturer_launch_vehicle_period': 5, 'spacecraft_l5_program_operator_launch_vehicle_period': 5}
template family counts: {'launch_vehi